[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/forestdatapartnership/whisp/blob/feat/timber-tree-rejig/notebooks/timber_pathway_viewer.ipynb)

# Whisp timber: interactive decision-tree / map viewer

This notebook builds an **interactive viewer** that links the Whisp **timber decision tree** (a Mermaid flowchart) to a **Leaflet map**, so you can explore how the timber risk pathway is built up from the underlying Earth Engine layers over Brazil.

**What it shows**
- Click a **2020 question** node to map that land class's 2020 source-agreement layer (how many input datasets agree the pixel is that class).
- Click a **2025 question** node to map the after-2020 gain / change layer.
- Click a **verdict terminal** to pick a feeding pathway code and map it in its palette colour.
- A **combined risk-outcome** map (green = low / amber = more-info / red = high) is available via the 'combined map' button (off by default), with solid/pale confidence shading.
- Toggles: **dilate** (zoom-stable bloom), **OR vs k>=2** agreement, Sentinel-2 2020 / 2024-25 before-after backgrounds, base map, and demo-site bookmarks.

Deforestation-risk frameworks such as the EU Deforestation Regulation (EUDR) are one example use; this is a general deforestation-risk view.

**How to run**
1. Open in Google Colab (badge above) or in a local Jupyter with the Whisp package installed.
2. Run the **Setup** cell to install `openforis-whisp` (it installs this **timber dev branch** from GitHub, since the viewer needs branch-only layers not yet on PyPI; after the branch merges, plain PyPI will work).
3. Set `PROJECT` to your own Earth Engine cloud project in the **first code cell at the top**, then run the **Earth Engine init** cell (it prompts you to authenticate the first time).
4. Run the **build** cell (resolves every map layer to a signed tile URL), then the **render** cell.

**Requirements:** a Google Earth Engine account and a registered cloud project. If unsure how to find your project id, see https://developers.google.com/earth-engine/cloud/assets

> **Note on tiles:** the Earth Engine tile URLs embedded in the map are *signed* and *expire* after some hours (Earth Engine does not publish an exact lifetime). This is inherent to `getMapId`. Just **re-run the build + render cells** to refresh them before a demo. The robustness here is about Earth Engine **initialisation**, not tile longevity.

## Set your Earth Engine project
Edit the one line in the cell below before running anything else.

In [1]:
# ==========================================================================
# >>> SET YOUR EARTH ENGINE PROJECT (the one thing you must edit) <<<
# Your Google Earth Engine cloud project id, e.g. "ee-yourname" or "my-gcp-project".
# Find or create one at https://code.earthengine.google.com (project switcher, top-left)
# and make sure the Earth Engine API is enabled. The init cell further down signs you in.
# ==========================================================================
PROJECT = "ee-andyarnellgee"

## Setup: install and import packages

In [2]:
# Install openforis-whisp. This viewer needs the DEV TIMBER BRANCH build (Ind_17_disturbance_after_2020_timber,
# the extra pathway codes, the timber_tree_export module), which is NOT on PyPI yet, so the default
# install is the GitHub branch. On a local checkout of that branch this is a no-op.
# NOTE the check imports timber_tree_export, not just openforis_whisp: a whisp from another branch
# carries the SAME version string, so pip reports "already satisfied" and silently skips the install.
BRANCH = 'feat/timber-tree-rejig'
REPO = 'git+https://github.com/forestdatapartnership/whisp.git@' + BRANCH
try:
    from openforis_whisp import timber_tree_export  # noqa: F401

    print('openforis-whisp timber-branch build already importable - nothing to install.')
except ImportError:
    import sys

    try:
        import openforis_whisp  # noqa: F401

        _stale = True
    except ImportError:
        _stale = False
    if _stale:
        print('A whisp WITHOUT timber_tree_export is installed; forcing the branch build over it...')
        !{sys.executable} -m pip install -q --force-reinstall --no-deps {REPO}
        print('\n>>> NOW RESTART THE RUNTIME (Colab: Runtime > Restart session), then run this cell again. <<<')
    else:
        !{sys.executable} -m pip install -q {REPO}
        print('Installed. If the build cell still fails, restart the runtime and re-run from the top.')
# after this branch merges, the released build will do:  !pip install --pre openforis-whisp

openforis-whisp already importable


In [3]:
# Choose the openforis-whisp build: UNCOMMENT ONE install line below, run this cell, then
# >>> RESTART THE KERNEL <<< so `import openforis_whisp` picks up the new build.
#
# Editable (local dev) and PyPI cannot coexist. If you are SWITCHING from one to the other,
# uncomment the uninstall line FIRST (run it), then uncomment your chosen install line.
import sys

# --- uninstall first (ONLY when switching between editable and PyPI) ---
# !{sys.executable} -m pip uninstall -y openforis-whisp

# (A) LOCAL DEV = this repo / branch: Ind_17, GPW, the missed-HIGH fix, codes 16/17/18; picks up your src edits.
#     Run from the repo ROOT. If your notebook's working dir is notebooks/, use  -e ..  instead of  -e .
# !{sys.executable} -m pip install -e .

# (B) RELEASED PyPI build (older tree until this branch is merged).
# !{sys.executable} -m pip install --pre openforis-whisp

# (C) THIS BRANCH from GitHub (remote / Colab, no local checkout).
# !{sys.executable} -m pip install git+https://github.com/forestdatapartnership/whisp.git@feat/timber-tree-rejig

print(">>> RESTART THE KERNEL after running an install line above, then run the notebook from the top. <<<")


>>> RESTART THE KERNEL after running an install line above, then run the notebook from the top. <<<


In [4]:
import json
import os
import ee
from IPython.display import HTML, display

## Earth Engine initialisation (robust, configurable)

Set `PROJECT` below to **your own** Earth Engine cloud project id. There is no hidden hardcoded project and no `.env` file: you control which project is used.

The init tries `ee.Initialize(project=PROJECT)`; if that fails it runs `ee.Authenticate()` and retries, and gives a clear error if it still cannot connect. (Set `PROJECT` to your own Earth Engine Cloud project id before running.)

In [5]:
# PROJECT is set at the TOP of the notebook (edit it there). The EE_PROJECT env var overrides it;
# globals().get keeps this cell runnable on its own if the top cell was not run.
PROJECT = os.environ.get('EE_PROJECT', globals().get('PROJECT', '<your_gee_cloud_project>'))

HIGH_VOLUME = 'https://earthengine-highvolume.googleapis.com'


def init_earthengine(project):
    """Robust, self-contained EE init: try Initialize, else Authenticate + retry, else explain."""
    if not project or project in ('your-ee-project-id', 'your_cloud_project_name', '<your_gee_cloud_project>'):
        raise ValueError(
            'Set PROJECT to your own Earth Engine cloud project id before running this cell.'
        )
    try:
        ee.Initialize(project=project, opt_url=HIGH_VOLUME)
    except Exception as first_err:  # noqa: BLE001
        print('First ee.Initialize failed (%s); authenticating...' % type(first_err).__name__)
        try:
            ee.Authenticate()
            ee.Initialize(project=project, opt_url=HIGH_VOLUME)
        except Exception as second_err:  # noqa: BLE001
            raise RuntimeError(
                "Earth Engine init failed for project '%s'.\n"
                'Check: (1) PROJECT is a cloud project YOU can access, '
                '(2) you completed ee.Authenticate() / are logged in, '
                '(3) the Earth Engine API is enabled for the project.\n'
                'Underlying error: %s' % (project, second_err)
            ) from second_err
    print('Earth Engine initialised on project:', project)


init_earthengine(PROJECT)

Earth Engine initialised on project: ee-andyarnellgee


## Build the viewer layers

This cell embeds the viewer logic (adapted from the internal build script, which lives in a git-ignored folder so it cannot be imported here). It reads the dataset lookup via `openforis_whisp.risk` and the per-dataset Earth Engine images via `openforis_whisp.datasets` (both shipped in the installable package), reconstructs the timber pathway, and resolves every map layer to a **signed XYZ tile template** via `getMapId(...)['tile_fetcher'].url_format`.

All layers are clipped to Brazil (the demo extent). This cell makes many Earth Engine calls, so it can take a minute or two.

In [6]:
# ============================================================================
# Whisp timber pathway viewer: layer build logic (embedded; self-contained).
# Adapted from the internal build script. Reads the lookup + datasets from the
# installed openforis-whisp package, rebuilds the timber pathway, and resolves
# each layer to a signed XYZ tile template (the tile token expires; re-run to refresh).
# ============================================================================
from openforis_whisp import datasets as d
from openforis_whisp.risk import lookup_gee_datasets_df as lut
import os as _os, openforis_whisp.risk as _risk
from openforis_whisp import timber_tree_export as tx  # single source: tree, codes, palette, walkers
from openforis_whisp import timber_map as tmap        # single source: the packaged pathway/outcome map builder

# This viewer is built for the DEV-BRANCH timber tree (07a/07b split, Ind_15/16, Ind_17, codes 16/17/18).
# If the installed whisp is the released / PyPI build (older tree), the branch-era getters are missing,
# so stop with a CLEAR instruction instead of a cryptic ImportError (see the version-chooser cell above).
_needed = [
    "get_cols_ind_01_treecover", "get_cols_ind_02_commodities", "get_cols_ind_04_dist_after_2020",
    "get_cols_ind_05_primary_2020", "get_cols_ind_06_nat_reg_2020", "get_cols_ind_07a_planted_2020",
    "get_cols_ind_07b_plantation_2020", "get_cols_ind_08b_plantation_after_2020",
    "get_cols_ind_09_treecover_after_2020", "get_cols_ind_10_agri_after_2020",
    "get_cols_ind_12_other_land_2020", "get_cols_ind_13_other_land_after_2020",
    "get_cols_ind_14_primary_2025", "get_cols_ind_15_agriculture_2020",
    "get_cols_ind_16_plantation_presence_2025",
]
_missing = [_n for _n in _needed if not hasattr(_risk, _n)]
if _missing:
    raise ImportError(
        "This timber-pathway viewer needs the DEV-BRANCH build of openforis-whisp. The installed "
        "package at " + _os.path.dirname(_risk.__file__) + " is the released / PyPI version (older "
        "timber tree) and is missing: " + ", ".join(_missing) + ".  FIX: run the version-chooser "
        "cell near the top (option A 'pip install -e .' for a local checkout, or option C the GitHub "
        "branch), RESTART THE KERNEL, then run the notebook from the top."
    )
globals().update({_n: getattr(_risk, _n) for _n in _needed})

brazil = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq("ADM0_NAME", "Brazil"))


def s2(start, end):  # Sentinel-2 true-colour median (ground reference under the risk/agreement layers)
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(brazil).filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20)).median()
    )


_S2_VIS = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}


# --- band / union / count helpers --------------------------------------------
def band_img(r):  # select by INDEX, global 0-fill so a limited-footprint member does not mask the union
    fn = getattr(d, str(r["corresponding_variable"]), None)
    if fn is None:
        return None
    try:
        return ee.Image(fn()).select(0).gt(0).unmask(0, False)
    except Exception:  # noqa: BLE001
        return None


def union(mask):
    imgs = [band_img(r) for _, r in lut[mask].iterrows() if band_img(r) is not None]
    if not imgs:
        return ee.Image(0)
    u = imgs[0]
    for im in imgs[1:]:
        u = u.Or(im)
    return u


def count(rows):
    imgs = [band_img(r) for r in rows if band_img(r) is not None]
    c = ee.Image(0)
    for im in imgs:
        c = c.add(im)
    return c


def flag(names):  # boolean mask over lut for rows whose name is in `names` (a whisp getter's result list)
    return lut["name"].isin(list(names))


# Resolve each whisp per-indicator getter once against the lookup (the names whisp feeds per indicator).
_n_ind01 = get_cols_ind_01_treecover(lut)
_n_ind02 = get_cols_ind_02_commodities(lut, risk_col="use_for_risk_pcrop")
_n_ind04 = get_cols_ind_04_dist_after_2020(lut)
_n_ind05 = get_cols_ind_05_primary_2020(lut)
_n_ind06 = get_cols_ind_06_nat_reg_2020(lut)
_n_ind07a = get_cols_ind_07a_planted_2020(lut)
_n_ind07b = get_cols_ind_07b_plantation_2020(lut)
_n_ind08b = get_cols_ind_08b_plantation_after_2020(lut)
_n_ind09 = get_cols_ind_09_treecover_after_2020(lut)
_n_ind10 = get_cols_ind_10_agri_after_2020(lut)
_n_ind12 = get_cols_ind_12_other_land_2020(lut)
_n_ind13 = get_cols_ind_13_other_land_after_2020(lut)
_n_ind14 = get_cols_ind_14_primary_2025(lut)
_n_ind15 = get_cols_ind_15_agriculture_2020(lut)
_n_ind16 = get_cols_ind_16_plantation_presence_2025(lut)

# Flag-wired pools (each built from exactly whisp's flagged input rows for the mapped indicator).
primary_rows = [r for _, r in lut[flag(_n_ind05)].iterrows()]
ag_rows = [r for _, r in lut[flag(_n_ind10)].iterrows()]
primary_count = count(primary_rows)
ag_count = count(ag_rows)

a = union(flag(_n_ind02) | flag(_n_ind15))             # Rule-1 agriculture-2020 (commodities OR agriculture_2020)
r20 = union(flag(_n_ind06)).Or(union(flag(_n_ind07a)))  # regen = nat-reg + planted
pl20 = union(flag(_n_ind07b))                          # plantation 2020
r25 = union(flag(_n_ind09))                            # treecover after 2020
pl25 = union(flag(_n_ind08b))                          # plantation gain (degradation)
pl25_presence = union(flag(_n_ind16))                  # multi-year plantation presence 2021-2024
ol25 = union(flag(_n_ind13))                           # other land after 2020
ol20 = union(flag(_n_ind12))                           # other land 2020
tc20 = union(flag(_n_ind01))                           # treecover gate (mirrors risk.py)
_dist_timber_mask = (
    (lut["use_for_risk_timber"] == 1)
    & (lut["theme_timber"] == "disturbance_after")
    & (lut["exclude_from_output"] != 1)
)
dist = union(_dist_timber_mask)                        # timber-specific disturbance after 2020
ind14 = union(flag(_n_ind14))                          # primary_2025 candidate (inert in pathway)


# --- the timber pathway-code image now comes from the packaged builder (single source) ---
# The hand-built qimg + tx.eval_tree_ee walk that used to live here is now
# openforis_whisp.timber_map.build_timber_pathway_image, assembled from the SAME get_cols_ind_* getters
# and the SAME derived combinations (forest_2020, primary_2025 = primary AND NOT disturbance,
# agriculture_2020) as risk.add_risk_timber_col. Proven histogram-identical to the old pathway() over
# Brazil, so the viewer is unchanged but can no longer drift from the tree or the tabular table.

# Palette, code names and verdict-class code sets all come from the single-source export module.
_PALETTE_BY_CODE = dict(tx.CODE_COLOUR)

_CODE_NAMES = tx.code_names()

# k=1 -> any single product fires the node; k=2 -> agreement (>=2 sources) on primary + agriculture.
# national_codes=["br"] reproduces the old full-lookup Brazil build; clip=False keeps it unbounded so the
# per-layer .clip(brazil) below is unchanged.
v_or = tmap.build_timber_pathway_image(region=brazil, national_codes=["br"], k=1, clip=False)
v_conv = tmap.build_timber_pathway_image(region=brazil, national_codes=["br"], k=2, clip=False)

# Verdict class-code sets: LOW / HIGH / more-info (derived from the tree via the export module).
_class_codes = tx.class_codes()
LOW_CODES = _class_codes["low"]
HIGH_CODES = _class_codes["high"]
MORE_CODES = _class_codes["more"]
_VERDICT_PALETTE = {"low": "41ab5d", "more": "f08c00", "high": "e31a1c"}
_VERDICT_VIS = {"min": 1, "max": 3, "palette": [_VERDICT_PALETTE["low"], _VERDICT_PALETTE["more"], _VERDICT_PALETTE["high"]]}


# The 14-code pathway image -> 1=LOW / 2=more-info / 3=HIGH collapse is now
# timber_map.collapse_to_outcome3 (same LOW / MORE / HIGH code sets, derived from tx.class_codes()).
verdict3_or = tmap.collapse_to_outcome3(v_or)
verdict3_conv = tmap.collapse_to_outcome3(v_conv)

# p20o (primary present, any source) is reused below by still_primary / regen_stayed / regen agreement.
p20o = primary_count.gte(1)

# --- zoom-stable dilation (fixed-scale reproject) ----------------------------
DILATE_KM = 4.0
_FIXED_M = 400
_DILATE_K = max(1, round(DILATE_KM * 1000 / _FIXED_M))


def _dilate(mask):  # focal_max over a fixed-scale circular kernel, constant at every zoom
    return (
        mask.selfMask()
        .focalMax(radius=_DILATE_K, kernelType="circle", units="pixels")
        .reproject(crs="EPSG:4326", scale=_FIXED_M)
    )


# --- 2020 agreement counts (flag-wired) --------------------------------------
_regen_rows = [r for _, r in lut[flag(_n_ind06) | flag(_n_ind07a)].iterrows()]
_plantation_rows = [r for _, r in lut[flag(_n_ind07b)].iterrows()]
_otherland_rows = [r for _, r in lut[flag(_n_ind12)].iterrows()]
_regen_count = count(_regen_rows)
_plantation_count = count(_plantation_rows)
_otherland_count = count(_otherland_rows)

# Split the 2020 agriculture agreement into cropland + tree-crop (pasture taken from the dedicated block).
_CROPLAND_NAMES = [
    "Soy_Song_2020", "nBR_INPE_TCamz_cer_annual_2020", "nBR_MapBiomas_col10_soy_2020",
    "nBR_MapBiomas_col10_annual_crops_2020", "GLAD_cropland_2020",
]
_TREECROP_NAMES = [
    "TMF_plant", "Oil_palm_Descals", "Oil_palm_FDaP", "Coffee_FDaP", "Cocoa_FDaP", "Cocoa_ETH",
    "Rubber_FDaP", "Rubber_RBGE", "ForTy_tree_crops_2020", "nCO_ideam_eufo_commission_2020",
    "nBR_INPE_TCamz_cer_perennial_2020", "nBR_MapBiomas_col10_coffee_2020",
    "nBR_MapBiomas_col10_palmoil_2020", "nBR_MapBiomas_col10_pc_2020", "nCI_Cocoa_bnetd",
]
_cropland_rows = [r for _, r in lut[lut["name"].isin(_CROPLAND_NAMES)].iterrows()]
_treecrop_rows = [r for _, r in lut[lut["name"].isin(_TREECROP_NAMES)].iterrows()]
_cropland_count = count(_cropland_rows)
_treecrop_count = count(_treecrop_rows)

# Pasture 2020 agreement: MapBiomas class 15 + INPE TerraClass + Global Pasture Watch.
_MB10 = ee.Image(
    "projects/mapbiomas-public/assets/brazil/lulc/collection10/mapbiomas_brazil_collection10_integration_v1")


def _m01p(img):
    return ee.Image(img).select(0).gt(0).unmask(0, False)


_p_mb_2020 = _m01p(_MB10.select("classification_2020").eq(15))
_p_tcamz = ee.Image("projects/ee-whisp/assets/NBR/terraclass_amz_2020")
_p_tccer = ee.Image("projects/ee-whisp/assets/NBR/terraclass_cer_2020")
_p_inpe_2020 = _m01p(_p_tcamz.eq(10).Or(_p_tcamz.eq(11))).Or(_m01p(_p_tccer.eq(11)))
_p_GPW = ee.ImageCollection("projects/global-pasture-watch/assets/ggc-30m/v1/grassland_c")
_p_gpw_dc = lambda yr: ee.Image(_p_GPW.filter(ee.Filter.eq("system:index", yr)).first()).select("dominant_class")
_p_gpw_2020 = _m01p(_p_gpw_dc("2020").eq(1))
_pasture_count = _p_mb_2020.add(_p_inpe_2020).add(_p_gpw_2020)
_p_mb_2024 = _m01p(_MB10.select("classification_2024").eq(15))
_p_gpw_2022 = _m01p(_p_gpw_dc("2022").eq(1))
_p_mb_gain = _p_mb_2024.And(_p_mb_2020.Not())
_p_gpw_gain = _p_gpw_2022.And(_p_gpw_2020.Not())
_pasture_gain = _p_mb_gain.Or(_p_gpw_gain)

# Agriculture as a whole 2020: each distinct ag source counts once, plus GPW (the one extra pasture source).
_ag2020_rows = [r for _, r in lut[flag(_n_ind02) | flag(_n_ind15)].iterrows()]
_ag2020_count = count(_ag2020_rows)
_ag_whole_count = _ag2020_count.add(_p_gpw_2020)
_AG_WHOLE_N = len(_ag2020_rows) + 1


def _row_img(name):
    return union(lut["name"] == name)


# After-2020 gain layers for the 2025 nodes.
_cropland_gain = _row_img("ESRI_crop_gain_2020_2025").Or(_row_img("GLAD_crop_gain_2020_2024"))
_treecrop_gain = _row_img("FDaP_tree_crop_gain_2020_2024")
_ag_whole_gain = union(flag(_n_ind10))                 # the deforestation `ag` union (Ind_10)
_plantation_gain = pl25
_other_land_gain = ol25.And(ol20.Not())
still_primary = p20o.And(dist.Not())
regen_stayed = r20.And(p20o.Not()).And(r25)

GAIN = {
    "cropland_gain": (_cropland_gain, "fe9929", "cropland gain after 2020 (annual/temporary; ESRI+GLAD)"),
    "treecrop_gain": (_treecrop_gain, "df65b0", "tree-crop gain after 2020 (perennial; FDaP palm/cocoa/rubber/coffee)"),
    "pasture_gain": (_pasture_gain, "78c679", "pasture gain after 2020 (MapBiomas 2020->2024 + GPW 2020->2022)"),
    "ag_whole_gain": (_ag_whole_gain, "d9880f", "agriculture gain after 2020 (all sources, the deforestation input)"),
    "plantation_gain": (_plantation_gain, "e7298a", "plantation expansion after 2020 (MapBiomas silviculture gain 2020->2024)"),
    "other_land_gain": (_other_land_gain, "ff7f00", "other-land gain after 2020 (new mining / built / water / rock / sand / salt)"),
    "plantation_presence": (pl25_presence, "41ab5d", "plantation presence 2021-2024 (still a plantation; MapBiomas silviculture any year)"),
    "forest_present_2025": (r25, "238b45", "forest present 2025 (treecover after 2020: TMF regrowth + ESRI 2025 treecover)"),
    "still_primary_2025": (still_primary, "08519c", "still primary 2025 (primary 2020 minus disturbance)"),
    "regen_stayed_2025": (regen_stayed, "7bccc4", "regen stayed forest 2025 (regen 2020 minus primary, still forest)"),
    "other_land_present_2025": (ol25, "7b6f5a", "other land present 2025 (other_land_after_2020 presence: ESRI 2025 + MapBiomas 2024 + mining after)"),
}

_RAMP2 = lambda c1, c2: {"min": 1, "max": 2, "palette": [c1, c2]}
_RAMP3 = lambda c1, c2, c3: {"min": 1, "max": 3, "palette": [c1, c2, c3]}

AGREEMENT = {
    "primary": (primary_count, _RAMP3("dbeecf", "74c476", "238b45"), len(primary_rows),
                "primary 2020 agreement (k>=1 / k>=2 / k>=3)"),
    "regen": (_regen_count.updateMask(p20o.Not()), _RAMP3("d7efd0", "7bccc4", "2b8cbe"), len(_regen_rows),
              "regen/planted 2020 agreement, EXCL primary (k>=1 / k>=2 / k>=3; matches the verdict carve-out)"),
    "plantation": (_plantation_count, _RAMP2("e5f5c9", "a1d99b"), len(_plantation_rows),
                   "plantation 2020 agreement (k>=1 / k>=2)"),
    "cropland": (_cropland_count, _RAMP3("fef0c8", "fdbb84", "d94801"), len(_cropland_rows),
                 "cropland 2020 agreement (annual/temporary; k>=1 / k>=2 / k>=3)"),
    "treecrop": (_treecrop_count, _RAMP3("f1e2cc", "c994c7", "980043"), len(_treecrop_rows),
                 "tree-crop 2020 agreement (perennial; k>=1 / k>=2 / k>=3)"),
    "pasture": (_pasture_count, _RAMP3("e5f0c0", "addd8e", "78c679"), 3,
                "pasture 2020 agreement (k>=1 / k>=2 / k>=3; MapBiomas+INPE+GPW)"),
    "ag_whole": (_ag_whole_count, {"min": 1, "max": _AG_WHOLE_N, "palette": ["f7e8c0", "e6b84d", "b8860b"]},
                 _AG_WHOLE_N, "agriculture-as-a-whole 2020 , count of distinct agriculture sources agreeing (cropland + tree-crop + pasture, incl. GPW)"),
    "other_land": (_otherland_count, _RAMP2("d8cdb8", "7b6f5a"), len(_otherland_rows),
                   "other land 2020 agreement (k>=1 / k>=2)"),
}


# --- resolve every layer to its signed XYZ tile template ---------------------
def _xyz(img, vis):  # getMapId -> the signed XYZ tile template (url_format). The token in this URL expires.
    return ee.Image(img).getMapId(vis)["tile_fetcher"].url_format


# --- PARALLELIZED tile-URL resolution ----------------------------------------
# getMapId is I/O-bound (one server round-trip per call) and EE's python client is thread-safe for it,
# so we resolve every (raw + dilated) tile template CONCURRENTLY via a thread pool instead of serially.
# We first DECLARE every job (a deferred ee image + vis under a unique slot id), resolve them all in
# parallel, then assemble the registry in the SAME order/keys as before from the resolved url map. This
# keeps the output config byte-for-byte equivalent to the serial build; only the resolution is parallel.
from concurrent.futures import ThreadPoolExecutor
import time as _time

_jobs = {}        # slot_id -> (ee_image, vis)  : every distinct tile template to resolve
_resolved = {}    # slot_id -> url             : filled in parallel below
_errors = {}      # slot_id -> exception       : surfaced after, never silently dropped


def _job(slot_id, img, vis):
    _jobs[slot_id] = (img, vis)
    return slot_id


# (1) Pathway layers for BOTH verdicts: code 1..14, raw + dilated, each in the code's palette colour.
_pathway_specs = []  # (key, hex, raw_slot, dil_slot, label)
_REACHED_CODES = sorted(_CODE_NAMES)  # exactly the codes the tree can emit (14 retired; 15 & 19 in use)
for _verdict_img, _prefix, _tag in [(v_conv, "pathconv", "k>=2"), (v_or, "pathor", "OR")]:
    for _code in _REACHED_CODES:
        _hex = _PALETTE_BY_CODE[_code]
        _mask = _verdict_img.eq(_code)
        _key = "%s_%d" % (_prefix, _code)
        _vis = {"palette": [_hex]}
        _rs = _job(_key + "::raw", _mask.selfMask().clip(brazil), _vis)
        _ds = _job(_key + "::dil", _dilate(_mask).clip(brazil), _vis)
        _pathway_specs.append((_key, _hex, _rs, _ds,
                               "pathway code %d (%s): %s" % (_code, _tag, _CODE_NAMES[_code])))

# (2) Agreement layers (graduated count), raw + dilated.
_agreement_specs = []  # (out_key, raw_slot, dil_slot, vis, label)
for _key, (_cimg, _vis, _n, _lab) in AGREEMENT.items():
    _shown = _cimg.updateMask(_cimg.gt(0))
    _rs = _job("agree_" + _key + "::raw", _shown.clip(brazil), _vis)
    _ds = _job("agree_" + _key + "::dil",
               _dilate(_cimg.gte(1)).multiply(_cimg).updateMask(_cimg.gt(0)).clip(brazil), _vis)
    _agreement_specs.append(("agree_" + _key, _rs, _ds, _vis, "%s [%d sources]" % (_lab, _n)))

# (3) Gain layers (after-2020 change), single-colour masks, raw + dilated.
_gain_specs = []  # (out_key, hex, raw_slot, dil_slot, label)
for _key, (_mask, _hex, _lab) in GAIN.items():
    _vis = {"palette": [_hex]}
    _m01 = ee.Image(_mask).gt(0)
    _rs = _job("gain_" + _key + "::raw", _m01.selfMask().clip(brazil), _vis)
    _ds = _job("gain_" + _key + "::dil", _dilate(_m01).clip(brazil), _vis)
    _gain_specs.append(("gain_" + _key, _hex, _rs, _ds, _lab))

# (3b) Verdict layers (default-on result view): plain solid 3-colour risk outcome, no pale/solid
# attenuation (green low / amber more-info / red high), raw + dilated.
_SOLID = {"low": "41ab5d", "more": "f08c00", "high": "e31a1c"}

_verdict_specs = []  # (out_key, raw_slot, dil_slot, label)
for _v3, _key, _tag in [(verdict3_or, "verdict_or", "OR"), (verdict3_conv, "verdict_conv", "k>=2")]:
    _rs = _job(_key + "::raw", _v3.clip(brazil), _VERDICT_VIS)
    _ds = _job(_key + "::dil", _dilate(_v3).clip(brazil), _VERDICT_VIS)
    _verdict_specs.append((_key, _rs, _ds,
                           "combined risk outcome (%s): green=low / amber=more-info / red=high" % _tag))

# (4) Sentinel-2 true-colour backgrounds (before / after).
_s2_specs = [
    ("s2_2020", _job("s2_2020", s2("2020-01-01", "2020-12-31").clip(brazil), _S2_VIS), "Sentinel-2 2020 (before)"),
    ("s2_2425", _job("s2_2425", s2("2024-06-01", "2025-12-31").clip(brazil), _S2_VIS), "Sentinel-2 2024-25 (after)"),
]

# ---- resolve EVERY declared job concurrently --------------------------------
def _resolve_one(slot_id):
    _img, _vis = _jobs[slot_id]
    return slot_id, _xyz(_img, _vis)


print("Resolving %d tile templates concurrently (getMapId, max_workers=16)..." % len(_jobs))
_t0 = _time.time()
with ThreadPoolExecutor(max_workers=16) as _ex:
    _fut_to_slot = {_ex.submit(_resolve_one, sid): sid for sid in _jobs}
    for _fut, _slot in _fut_to_slot.items():
        try:
            _sid, _u = _fut.result()
            _resolved[_sid] = _u
        except Exception as _e:  # noqa: BLE001 -- record per slot, do not let one failure kill the build
            _errors[_slot] = repr(_e)
# A failed resolve leaves its id out of _resolved; surface every such slot + its error (never silent).
if _errors:
    print("WARNING: %d tile template(s) failed to resolve:" % len(_errors))
    for _slot, _msg in _errors.items():
        print("  %s -> %s" % (_slot, _msg))
print("Resolved %d/%d tile templates in %.1fs." % (len(_resolved), len(_jobs), _time.time() - _t0))


def _url(slot_id):  # fetch a resolved url; raise a clear error if a needed slot failed (not silent)
    if slot_id not in _resolved:
        raise RuntimeError("tile template '%s' failed to resolve via getMapId (see warnings above)" % slot_id)
    return _resolved[slot_id]


# ---- assemble the registry in the SAME order / keys / structure as the serial build ----
registry = {}
for _key, _hex, _rs, _ds, _label in _pathway_specs:
    registry[_key] = {"raw_url": _url(_rs), "dilated_url": _url(_ds), "colour": _hex,
                      "kind": "pathway", "label": _label}
for _out_key, _rs, _ds, _vis, _label in _agreement_specs:
    registry[_out_key] = {"raw_url": _url(_rs), "dilated_url": _url(_ds), "colour": _vis["palette"][-1],
                          "kind": "agreement", "label": _label, "vis": _vis}
for _out_key, _hex, _rs, _ds, _label in _gain_specs:
    registry[_out_key] = {"raw_url": _url(_rs), "dilated_url": _url(_ds), "colour": _hex,
                          "kind": "gain", "label": _label}
for _out_key, _rs, _ds, _label in _verdict_specs:
    registry[_out_key] = {"raw_url": _url(_rs), "dilated_url": _url(_ds), "colour": _SOLID["high"],
                          "kind": "verdict", "label": _label}
_STATS = {}  # build-time stats panel removed (the area sum hit the Image.reproject limit)

S2_BACKGROUNDS = {_k: (_url(_slot), _lab) for _k, _slot, _lab in _s2_specs}

print("Resolved %d overlay layers + %d S2 backgrounds." % (len(registry), len(S2_BACKGROUNDS)))

Resolving 116 tile templates concurrently (getMapId, max_workers=16)...


2026-07-31 14:49:40,418 - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2026-07-31 14:49:40,910 - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2026-07-31 14:49:40,993 - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2026-07-31 14:49:41,146 - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2026-07-31 14:49:47,994 - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2026-07-31 14:49:51,214 - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10


Resolved 116/116 tile templates in 19.1s.
Resolved 57 overlay layers + 2 S2 backgrounds.


In [7]:
# ============================================================================
# Curated per-DATASET viz overlays (Part 1 + Part 2) + the question -> indicator
# -> datasets reference data (Part 1b).  VISUALISATION ONLY: these become
# independent toggle overlays (many on at once); they NEVER change the outcome
# or the drawn-plot table.  Reuses the build-cell helpers (band_img, _xyz, lut,
# ThreadPoolExecutor, brazil, tx) already in scope from the cell above.
# ============================================================================
import re as _re

# --- indicator column name -> the dataset names whisp feeds it (from the build cell's getters) ---
_disturbance_names = list(lut[_dist_timber_mask]["name"])
_IND_COL_TO_NAMES = {
    "Ind_01_treecover": list(_n_ind01),
    "Ind_02_commodities": list(_n_ind02),
    "Ind_05_primary_2020": list(_n_ind05),
    "Ind_06_nat_reg_forest_2020": list(_n_ind06),
    "Ind_07a_planted_2020": list(_n_ind07a),
    "Ind_07b_plantation_2020": list(_n_ind07b),
    "Ind_08b_plantation_after_2020": list(_n_ind08b),
    "Ind_09_treecover_after_2020": list(_n_ind09),
    "Ind_10_agri_after_2020": list(_n_ind10),
    "Ind_12_other_land_2020": list(_n_ind12),
    "Ind_13_other_land_after_2020": list(_n_ind13),
    "Ind_15_agriculture_2020": list(_n_ind15),
    "Ind_16_plantation_presence_2025": list(_n_ind16),
    "Ind_17_disturbance_after_2020_timber": _disturbance_names,
}

# The indicators the timber tree uses, in a sensible display order, each with a friendly group name.
_USED_INDICATORS = [
    ("Ind_01_treecover", "treecover 2020"),
    ("Ind_05_primary_2020", "primary 2020"),
    ("Ind_06_nat_reg_forest_2020", "naturally regenerating 2020"),
    ("Ind_07a_planted_2020", "planted 2020"),
    ("Ind_07b_plantation_2020", "plantation 2020"),
    ("Ind_02_commodities", "commodities 2020"),
    ("Ind_15_agriculture_2020", "agriculture 2020"),
    ("Ind_10_agri_after_2020", "agriculture after 2020"),
    ("Ind_08b_plantation_after_2020", "plantation gain after 2020"),
    ("Ind_16_plantation_presence_2025", "plantation presence 2021-2024"),
    ("Ind_09_treecover_after_2020", "treecover after 2020"),
    ("Ind_17_disturbance_after_2020_timber", "disturbance after 2020 (timber)"),
    ("Ind_12_other_land_2020", "other land 2020"),
    ("Ind_13_other_land_after_2020", "other land after 2020"),
]
_USED_PALETTE = ["1f78b4", "33a02c", "e31a1c", "ff7f00", "6a3d9a", "b15928", "a6cee3",
                 "b2df8a", "fb9a99", "fdbf6f", "cab2d6", "8dd3c7", "1b9e77", "d95f02"]

# --- Part 2: the "context" datasets (attached to an indicator theme but used_for_risk == 0 everywhere),
#     EXCLUDING the per-year time series.  No is_timeseries flag exists in the lookup yet, so identify the
#     series by a shared corresponding_variable (>1 row) EXCEPT g_glad_gfc_10pc_prep, which is two distinct
#     representative treecover layers, not a series.  (A dedicated is_timeseries CSV flag is the cleaner
#     long-term fix, noted in the plan; done in-notebook here to avoid touching the shared lookup + tests.)
_TS_KEEP_PREPS = {"g_glad_gfc_10pc_prep"}
_cv_counts = lut["corresponding_variable"].value_counts()


def _is_timeseries(row):
    cv = str(row.get("corresponding_variable"))
    return (_cv_counts.get(cv, 0) > 1) and (cv not in _TS_KEEP_PREPS)


def _grp(row):  # group label: theme, falling back to theme_timber
    for _k in ("theme", "theme_timber"):
        _v = row.get(_k)
        if _v is not None and str(_v).strip() not in ("", "nan", "NA", "None"):
            return str(_v).strip()
    return "other"


_nonrisk_mask = (
    (lut["use_for_risk_pcrop"] != 1)
    & (lut["use_for_risk_acrop"] != 1)
    & (lut["use_for_risk_timber"] != 1)
    & (lut["exclude_from_output"] != 1)
    & (lut["theme"].astype(str) != "context_and_metadata")
)
_context_rows = [r for _, r in lut[_nonrisk_mask].iterrows() if not _is_timeseries(r)]

# --- assign each dataset a colour + a group BEFORE resolving (getMapId needs the palette) ---
_ds_colour = {}   # name -> hex
_ds_group = {}    # name -> ("used"|"context", group_key, friendly_label)
for _i, (_col, _friendly) in enumerate(_USED_INDICATORS):
    _c = _USED_PALETTE[_i % len(_USED_PALETTE)]
    for nm in _IND_COL_TO_NAMES.get(_col, []):
        if nm not in _ds_colour:
            _ds_colour[nm] = _c
            _ds_group[nm] = ("used", _col, _friendly)
_CTX_GREYS = ["9e9e9e", "757575", "bdbdbd", "616161", "cfcfcf", "8a8a8a"]
_ctx_theme_order = []
for r in _context_rows:
    nm = r["name"]
    if nm in _ds_colour:  # already shown as a used-in-risk layer; do not duplicate in context
        continue
    _t = _grp(r)
    if _t not in _ctx_theme_order:
        _ctx_theme_order.append(_t)
    _ds_colour[nm] = _CTX_GREYS[_ctx_theme_order.index(_t) % len(_CTX_GREYS)]
    _ds_group[nm] = ("context", _t, _t)


def _row_by_name(nm):
    _m = lut[lut["name"] == nm]
    return _m.iloc[0] if len(_m) else None


# build the (image, vis) job per dataset that actually resolves to a band image
_ds_jobs = {}
for nm in _ds_colour:
    _row = _row_by_name(nm)
    _img = band_img(_row) if _row is not None else None
    if _img is not None:
        _ds_jobs[nm] = (_img.selfMask().clip(brazil), {"palette": [_ds_colour[nm]]})


def _resolve_ds(nm):
    _img, _vis = _ds_jobs[nm]
    return nm, _xyz(_img, _vis)


print("Resolving %d curated dataset tiles concurrently (getMapId, max_workers=16)..." % len(_ds_jobs))
_ds_url = {}
_ds_err = {}
with ThreadPoolExecutor(max_workers=16) as _ex:
    _futs = {_ex.submit(_resolve_ds, nm): nm for nm in _ds_jobs}
    for _fut, _nm in _futs.items():
        try:
            _rn, _u = _fut.result()
            _ds_url[_rn] = _u
        except Exception as _e:  # noqa: BLE001 -- record per layer, never abort the whole build
            _ds_err[_nm] = repr(_e)
if _ds_err:
    print("WARNING: %d dataset tile(s) failed to resolve (skipped):" % len(_ds_err))
    for _nm, _msg in list(_ds_err.items())[:12]:
        print("  %s -> %s" % (_nm, _msg))
print("Resolved %d/%d dataset tiles." % (len(_ds_url), len(_ds_jobs)))

DATASET_LAYERS = {nm: {"url": _ds_url[nm], "colour": _ds_colour[nm]} for nm in _ds_url}

# --- ordered groups for the layer UI (each split into global + national members) ---
_ISO_NAME = {"BR": "Brazil", "CO": "Colombia", "CI": "Cote d'Ivoire", "CM": "Cameroon"}


def _iso_of(nm):
    _r = _row_by_name(nm)
    if _r is None:
        return None
    _v = str(_r.get("ISO2_code")).strip()
    return _v if _v and _v.lower() != "nan" else None


def _split_national(names):
    _g, _nat, _isos = [], [], []
    for nm in names:
        _iso = _iso_of(nm)
        if _iso:
            _nat.append(nm)
            if _iso not in _isos:
                _isos.append(_iso)
        else:
            _g.append(nm)
    _lab = ("national (%s)" % ", ".join(_ISO_NAME.get(i, i) for i in _isos)) if _nat else "national"
    return _g, _nat, _lab


_used_groups = []
for _col, _friendly in _USED_INDICATORS:
    _members = [nm for nm in _IND_COL_TO_NAMES.get(_col, [])
                if nm in DATASET_LAYERS and _ds_group[nm][:2] == ("used", _col)]
    if _members:
        _g, _nat, _lab = _split_national(_members)
        _used_groups.append({"indicator": _col, "label": "%s  (%s)" % (_col, _friendly),
                             "global": _g, "national": _nat, "national_label": _lab})
_ctx_map = {}
for r in _context_rows:
    nm = r["name"]
    if nm in DATASET_LAYERS and _ds_group.get(nm, (None,))[0] == "context":
        _ctx_map.setdefault(_grp(r), []).append(nm)
_context_groups = []
for _t, _ds in _ctx_map.items():
    _g, _nat, _lab = _split_national(_ds)
    _context_groups.append({"theme": _t, "label": _t, "global": _g, "national": _nat, "national_label": _lab})
DATASET_GROUPS = {"used": _used_groups, "context": _context_groups}


# --- Part 1b: question -> indicator(s) -> datasets reference rows (from tx.Q_LABEL + tx.Q_TO_COLUMNS) ---
def _strip_html(s):
    return _re.sub(r"\s+", " ", _re.sub(r"<[^>]+>", " ", str(s))).strip()


REFERENCE_ROWS = []
for _q, _cols in tx.Q_TO_COLUMNS.items():
    _label = _strip_html(tx.Q_LABEL.get(_q, _q))
    if list(_cols) == ["primary_2025"]:  # derived column: show its real indicators
        _inds = ["Ind_05_primary_2020", "Ind_17_disturbance_after_2020_timber"]
        _expr = "Ind_05_primary_2020 AND NOT Ind_17_disturbance_after_2020_timber (derived)"
    else:
        _inds = list(_cols)
        _expr = " OR ".join(_cols)
    _dsn = []
    for _c in _inds:
        for nm in _IND_COL_TO_NAMES.get(_c, []):
            if nm not in _dsn:
                _dsn.append(nm)
    REFERENCE_ROWS.append({"question": _label, "expr": _expr, "datasets": _dsn})

print("Dataset overlays: %d used-in-risk group(s), %d context group(s); %d reference rows." % (
    len(_used_groups), len(_context_groups), len(REFERENCE_ROWS)))

Resolving 95 curated dataset tiles concurrently (getMapId, max_workers=16)...


2026-07-31 14:49:55,962 - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2026-07-31 14:49:56,062 - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2026-07-31 14:49:56,096 - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2026-07-31 14:49:56,145 - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2026-07-31 14:49:56,348 - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2026-07-31 14:49:56,496 - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10


Resolved 95/95 dataset tiles.
Dataset overlays: 14 used-in-risk group(s), 10 context group(s); 12 reference rows.


## Assemble the interactive HTML (Mermaid tree + Leaflet map)

This builds the Mermaid decision-tree source and the Leaflet + Mermaid HTML scaffold, then injects the resolved tile URLs and node/terminal mappings into it.

In [8]:
# ============================================================================
# Mermaid decision-tree source + node/terminal mappings + the Leaflet+Mermaid
# HTML scaffold. The resolved tile URLs (registry / S2_BACKGROUNDS) are injected
# into the scaffold as a JSON CONFIG object.
# ============================================================================

from openforis_whisp import timber_tree_export as tx  # single source: tree, codes, mermaid, JS walk

# NODE id -> drill-down map layer(s), derived from the tree (per-question default + per-branch overrides).
NODE_TO_LAYER = tx.node_to_layer()
AGREEMENT_OPTION_LABELS = {
    "agree_ag_whole": "agriculture as a whole (distinct sources)",
    "agree_cropland": "cropland (annual/temporary)",
    "agree_treecrop": "tree-crop (perennial)",
    "agree_pasture": "pasture",
    "gain_cropland_gain": "cropland gain (ESRI+GLAD)",
    "gain_treecrop_gain": "tree-crop gain (FDaP)",
    "gain_pasture_gain": "pasture gain (MapBiomas+GPW)",
    "gain_ag_whole_gain": "agriculture as a whole",
}
# terminal node id (term_<code>) -> (code, verdict-class), built from the tree's leaves.
TERMINAL_TO_CODE = {}
for _cls, _codes in tx.class_codes().items():
    for _c in _codes:
        TERMINAL_TO_CODE["term_%d" % _c] = (_c, _cls)
CLASS_CODES = {"low": LOW_CODES, "high": HIGH_CODES, "more": MORE_CODES}

CODE_TO_TERMINALS = {}
for _tid, (_c, _v) in TERMINAL_TO_CODE.items():
    CODE_TO_TERMINALS.setdefault(str(_c), []).append(_tid)

CODE_LABELS = {str(c): _CODE_NAMES[c] for c in _CODE_NAMES}

BOOKMARKS = [
    ["Tocantinzinho post-2020 mine", -6.0519, -56.2935, 13],
    ["Pasture expansion (Amazon basin)", -6.4478, -56.1887, 13],
    ["Southern silviculture belt", -22.0, -48.0, 7],
    ["Cerrado pasture-gain hotspot (Matopiba)", -9.5, -45.5, 8],
]

# --- Mermaid timber decision-tree source (generated from risk.TIMBER_ROOT_TREE) ---
MERMAID_SRC = tx.to_mermaid()

_decision_ids, _terminal_ids = tx.node_ids()

_JS_CONFIG = {
    "registry": registry,
    "s2Backgrounds": {k: {"url": u, "label": lab} for k, (u, lab) in S2_BACKGROUNDS.items()},
    "nodeToLayer": NODE_TO_LAYER,
    "agreementOptionLabels": AGREEMENT_OPTION_LABELS,
    "terminalToCode": {k: {"code": v[0], "verdict": v[1]} for k, v in TERMINAL_TO_CODE.items()},
    "codeToTerminals": CODE_TO_TERMINALS,
    "classCodes": CLASS_CODES,
    "codeLabels": CODE_LABELS,
    "bookmarks": BOOKMARKS,
    "palette": _PALETTE_BY_CODE,
    "pathwayLabelToCode": tx.pathway_label_to_code(),
    "allNodes": _decision_ids,
    "allTerminals": _terminal_ids,
    "verdictLayers": {"or": "verdict_or", "conv": "verdict_conv"},
    "stats": _STATS,
    "verdictSwatch": {"low": _SOLID["low"], "more": _SOLID["more"], "high": _SOLID["high"]},
    "datasetLayers": DATASET_LAYERS,
    "datasetGroups": DATASET_GROUPS,
    "referenceRows": REFERENCE_ROWS,
}

# --- the Leaflet + Mermaid HTML scaffold -------------------------------------
HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8" />
<meta name="viewport" content="width=device-width, initial-scale=1" />
<title>WHISP timber: interactive decision-tree / map viewer</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<!-- Leaflet.draw: the polygon-draw control for the "draw -> whisp -> colour by risk" feature. -->
<link rel="stylesheet" href="https://unpkg.com/leaflet-draw@1.0.4/dist/leaflet.draw.css" />
<script src="https://unpkg.com/leaflet-draw@1.0.4/dist/leaflet.draw.js"></script>
<style>
  html, body { height: 100%; }
  body { font-family: -apple-system, Segoe UI, Helvetica, Arial, sans-serif; margin: 0; color: #1f2937; }
  header { padding: .7rem 1.1rem; background: #166534; color: #fff; }
  header h1 { font-size: 1.05rem; margin: 0; }
  header p { margin: .25rem 0 0; font-size: .78rem; opacity: .92; }
  .wrap { display: flex; height: calc(100vh - 64px); min-height: 760px; }
  #treepane { width: 46%; min-width: 200px; min-height: 760px; overflow: auto; border-right: 1px solid #e5e7eb; padding: .8rem; background: #fafafa; flex: 0 0 auto; }
  #splitter { flex: 0 0 6px; cursor: col-resize; background: #e5e7eb; border-left: 1px solid #d1d5db;
              border-right: 1px solid #d1d5db; }
  #splitter:hover { background: #cbd5e1; }
  body.dragging { cursor: col-resize; user-select: none; }
  #treepane svg { max-width: 100%; height: auto; }
  .node-clickable { cursor: pointer; }
  g.node-active rect, g.node-active polygon, g.node-active circle, g.node-active path {
    stroke: #ff6d00 !important; stroke-width: 5px !important;
    filter: drop-shadow(0 0 6px #ff6d00) drop-shadow(0 0 3px #1d4ed8);
  }
  g.node-active { filter: drop-shadow(0 0 8px rgba(255,109,0,.9)); }
  #mappane { flex: 1; min-height: 760px; position: relative; }
  #controls { padding: .6rem .7rem; font-size: .8rem; }
  #controls > * { display: block; margin: 0 0 .6rem 0; }
  #controls select, #controls button { font-size: .8rem; padding: .2rem .35rem; }
  #controls .bm button { margin-left: .25rem; }
  #map { width: 100%; height: 100%; min-height: 760px; }
  #controlspane { flex: 0 0 240px; overflow-y: auto; background: #f3f4f6; border-left: 1px solid #e5e7eb; }
  #status { font-size: .78rem; padding: .35rem .8rem; background: #fff; border-bottom: 1px solid #e5e7eb; min-height: 1.1rem; }
  #legend { position: absolute; bottom: 16px; right: 12px; z-index: 1000; background: #fff;
            padding: 6px 9px; font-size: 11px; border: 1px solid #999; border-radius: 5px;
            box-shadow: 0 1px 4px rgba(0,0,0,.3); max-width: 250px; }
  #legend .sw { display: inline-block; width: 12px; height: 12px; border: 1px solid #777;
                margin-right: 5px; vertical-align: middle; }
  .hint { color: #6b7280; }
  #datasetBody label, #datasetBody summary { border-radius:2px; }
  #datasetBody label:hover, #datasetBody summary:hover { background:#e5e7eb; }
</style>
</head>
<body>
<header>
  <h1>WHISP timber: decision tree linked to the map
      <span id="helpToggle" title="What can I click? (click for help)" onclick="var p=document.getElementById('helpPanel'); p.style.display=(p.style.display==='none'?'block':'none');" style="cursor:pointer; font-size:0.6em; color:#2563eb; user-select:none; vertical-align:middle;">&#9432;</span></h1>
  <div id="helpPanel" class="hint" style="display:none; margin-top:6px; max-width:70ch;">
  A demo map showing the timber decision tree in map form. Click any box in the tree to see the
     data behind that step, and click an outcome to see what drove it. Switch the base map (plain
     grey or satellite), and toggle the satellite imagery on beneath the layers. It is an exploration
     tool for deforestation risk, not a final decision; EUDR is one example of where it can help.</div>
</header>

<div class="wrap">
  <div id="treepane">rendering tree...</div>
  <div id="splitter" title="drag to resize the tree / map panes"></div>
  <div id="mappane">
    <div id="map"></div>
    <div id="legend"><b>Legend</b><div id="legendBody" class="hint">select a layer</div></div>
  </div>
  <div id="controlspane">
    <div id="controls">
      <span><b id="curlabel">no layer</b></span>
      <button id="verdictBtn" title="show the combined risk-outcome map (low / more-info / high)">combined map</button>
      <label class="hint">layer:
        <select id="codeSelect" disabled></select></label>
      <label class="hint">agreement:
        <select id="verdictSelect" title="Experimental. k>=2 requires >=2 agreeing products, but only on the primary-2020 and deforestation nodes; all other nodes stay OR and single-source pixels drop out.">
          <option value="or" selected>OR (any 1 product)</option>
          <option value="conv">k&gt;=2 agreement (experimental)</option>
        </select></label>
      <label><input type="checkbox" id="dilateChk"> dilate (zoom-stable)</label>
      <label><input type="checkbox" id="mainLayerChk" checked> show layer</label>
      <span class="hint">base:
        <label><input type="radio" name="base" value="positron" checked> grey OSM</label>
        <label><input type="radio" name="base" value="esri"> Esri</label></span>
      <span class="hint">Sentinel-2:
        <label><input type="checkbox" id="s2_2020chk"> 2020</label>
        <label><input type="checkbox" id="s2_2425chk"> 2024-25</label></span>
      <span class="bm hint">go to:<span id="bookmarks"></span></span>
      <button id="clearBtn">clear</button>
    </div>
    <div id="status" class="hint">Click 'combined map' for the combined risk-outcome map (green=low / amber=more-info / red=high), or a node for the drill-down.</div>
      <details id="datasetPanel" style="margin-top:.4rem;border-top:1px solid #e5e7eb;padding:.4rem .55rem .2rem;">
        <summary style="font-weight:600;cursor:pointer;font-size:.8rem;">Dataset layers (visualise)</summary>
        <div class="hint" style="font-size:.68rem;margin:.2rem 0;color:#4b5563;">Independent overlays for context. Toggling these changes only the map, not the outcome.</div>
        <div id="datasetBody" style="font-size:.72rem;margin-top:.3rem;"></div>
      </details>
      <details id="refPanel" style="margin-top:.4rem;border-top:1px solid #e5e7eb;padding:.4rem .55rem .2rem;">
        <summary style="font-weight:600;cursor:pointer;font-size:.8rem;">Tree questions -&gt; indicators -&gt; datasets</summary>
        <div id="refBody" style="font-size:.72rem;margin-top:.3rem;"></div>
      </details>
  </div>
</div>

<script type="module">
import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs';
mermaid.initialize({ startOnLoad: false, securityLevel: 'loose', flowchart: { htmlLabels: true, rankSpacing: 80 } });

const CONFIG = __CONFIG__;
const MERMAID_SRC = __MERMAID__;

const map = L.map('map', { center: [-14, -50], zoom: 4 });
map.createPane('s2lo');  map.getPane('s2lo').style.zIndex  = 240;
map.createPane('s2hi');  map.getPane('s2hi').style.zIndex  = 260;
map.createPane('ovpane'); map.getPane('ovpane').style.zIndex = 450;

const BASES = {
  esri: L.tileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    { maxZoom: 19, attribution: 'Esri World Imagery' }),
  positron: L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}.png',
    { maxZoom: 19, subdomains: 'abcd', attribution: '&copy; OpenStreetMap, &copy; CARTO' }),
};
let currentBase = 'positron';
BASES.positron.addTo(map);
function setBase(value) {
  if (value === currentBase || !BASES[value]) return;
  map.removeLayer(BASES[currentBase]);
  BASES[value].addTo(map);
  BASES[value].bringToBack();
  currentBase = value;
}
document.querySelectorAll('input[name="base"]').forEach(r =>
  r.addEventListener('change', e => { if (e.target.checked) setBase(e.target.value); }));

const S2 = {
  s2_2020: L.tileLayer(CONFIG.s2Backgrounds.s2_2020.url, { pane: 's2lo', maxZoom: 19, attribution: CONFIG.s2Backgrounds.s2_2020.label }),
  s2_2425: L.tileLayer(CONFIG.s2Backgrounds.s2_2425.url, { pane: 's2hi', maxZoom: 19, attribution: CONFIG.s2Backgrounds.s2_2425.label }),
};
document.getElementById('s2_2020chk').addEventListener('change', e => {
  if (e.target.checked) S2.s2_2020.addTo(map); else map.removeLayer(S2.s2_2020);
});
document.getElementById('s2_2425chk').addEventListener('change', e => {
  if (e.target.checked) S2.s2_2425.addTo(map); else map.removeLayer(S2.s2_2425);
});

const LAYERS = {};
for (const [key, meta] of Object.entries(CONFIG.registry)) {
  LAYERS[key] = {
    raw: L.tileLayer(meta.raw_url, { pane: 'ovpane', opacity: 0.85, maxZoom: 19 }),
    dilated: L.tileLayer(meta.dilated_url, { pane: 'ovpane', opacity: 0.85, maxZoom: 19 }),
    meta: meta,
  };
}

let verdictMode = 'or';
function pathKey(code) { return (verdictMode === 'conv' ? 'pathconv_' : 'pathor_') + code; }
function verdictKey() { return CONFIG.verdictLayers[verdictMode]; }

const INDEPENDENT_KEYS = [];

let currentKey = null;
function dilateOn() { return document.getElementById('dilateChk').checked; }
function mainLayerOn() { return document.getElementById('mainLayerChk').checked; }

function hideAll() {
  for (const k in LAYERS) {
    if (INDEPENDENT_KEYS.includes(k)) continue;
    map.removeLayer(LAYERS[k].raw); map.removeLayer(LAYERS[k].dilated);
  }
}
function showLayer(key) {
  hideAll();
  currentKey = key;
  if (!key) { setStatus('No layer shown. Click a node or a verdict.'); setLegend(null); return; }
  // FIX: guard a missing/renamed registry key. If the layer does not exist (e.g. a verdict key that
  // drifted out of CONFIG.verdictLayers / the registry), surface it instead of dereferencing undefined and
  // throwing, which previously aborted showLayer silently and left the map un-repainted.
  const lyr = LAYERS[key];
  if (!lyr) {
    currentKey = null;
    setStatus('layer "' + key + '" is not registered (no tiles to show)');
    setLegend(null);
    console.warn('showLayer: no LAYERS entry for key', key, '- known keys:', Object.keys(LAYERS));
    return;
  }
  if (mainLayerOn()) {
    const tl = (dilateOn() ? lyr.dilated : lyr.raw);
    tl.addTo(map);
    // FIX: force a tile refresh on (re-)add. Re-adding the SAME cached Leaflet tile-layer object (the
    // verdict layer is re-added every time the "final verdict" button is pressed after navigating away)
    // could leave Leaflet serving its already-loaded tile set without re-requesting for the current view,
    // so the map appeared NOT to repaint. redraw() re-fetches the tiles for the active viewport.
    if (typeof tl.redraw === 'function') tl.redraw();
  }
  document.getElementById('curlabel').textContent = lyr.meta.label;
  setStatus(lyr.meta.label + (dilateOn() ? '  (dilated)' : '  (raw 30 m)'));
  setLegend(key);
}
function showVerdict() {
  dropdownMode = null;
  const sel = document.getElementById('codeSelect'); sel.disabled = true; sel.innerHTML = '';
  showLayer(verdictKey());
}
function setStatus(t) { document.getElementById('status').textContent = t; }

function setLegend(key) {
  const body = document.getElementById('legendBody');
  if (!key) { body.innerHTML = 'select a layer'; return; }
  const meta = LAYERS[key].meta;
  if (meta.kind === 'verdict') {
    const sw = CONFIG.verdictSwatch;
    body.innerHTML = "<div style='margin-bottom:3px'>combined risk outcome (" + verdictMode.toUpperCase() + ")</div>" +
      "<div><span class='sw' style='background:#" + sw.low + "'></span>low</div>" +
      "<div><span class='sw' style='background:#" + sw.more + "'></span>more info</div>" +
      "<div><span class='sw' style='background:#" + sw.high + "'></span>high</div>";
  } else if (meta.kind === 'agreement') {
    const pal = meta.vis.palette;
    body.innerHTML = "<div style='margin-bottom:3px'>" + meta.label + "</div>" +
      pal.map((c, i) => "<div><span class='sw' style='background:#" + c + "'></span>k&gt;=" + (i + 1) + "</div>").join('');
  } else {
    body.innerHTML = "<div><span class='sw' style='background:#" + meta.colour + "'></span>" + meta.label + "</div>";
  }
}

function renderStats() { /* stats panel removed */ }

document.getElementById('dilateChk').addEventListener('change', () => { if (currentKey) showLayer(currentKey); });
document.getElementById('mainLayerChk').addEventListener('change', () => { if (currentKey) showLayer(currentKey); });
document.getElementById('clearBtn').addEventListener('click', () => {
  showLayer(null);
  const sel = document.getElementById('codeSelect'); sel.disabled = true; sel.innerHTML = ''; dropdownMode = null;
  document.getElementById('curlabel').textContent = 'no layer';
  if (typeof _activeNode !== 'undefined' && _activeNode) { _activeNode.classList.remove('node-active'); _activeNode = null; }
  if (typeof clearActiveSet === 'function') clearActiveSet();
});

const codeSelect = document.getElementById('codeSelect');
let dropdownMode = null;
function populateCodes(codes, selectedCode) {
  codeSelect.innerHTML = '';
  codes.forEach(c => {
    const opt = document.createElement('option');
    opt.value = String(c);
    opt.textContent = 'code ' + c + ': ' + CONFIG.codeLabels[String(c)];
    if (c === selectedCode) opt.selected = true;
    codeSelect.appendChild(opt);
  });
  codeSelect.disabled = false; dropdownMode = 'pathway';
}
function populateSubgroup(layerKeys, selectedKey) {
  codeSelect.innerHTML = '';
  layerKeys.forEach(k => {
    const opt = document.createElement('option');
    opt.value = k;
    opt.textContent = CONFIG.agreementOptionLabels[k] || (LAYERS[k] ? LAYERS[k].meta.label : k);
    if (k === selectedKey) opt.selected = true;
    codeSelect.appendChild(opt);
  });
  codeSelect.disabled = false; dropdownMode = 'subgroup';
}
codeSelect.addEventListener('change', () => {
  if (dropdownMode === 'pathway') {
    showLayer(pathKey(codeSelect.value));
    highlightTerminals(CONFIG.codeToTerminals[String(codeSelect.value)]);
  } else {
    showLayer(codeSelect.value);
  }
});

document.getElementById('verdictSelect').addEventListener('change', e => {
  const showingVerdict = (currentKey === CONFIG.verdictLayers.or || currentKey === CONFIG.verdictLayers.conv);
  verdictMode = e.target.value;
  renderStats();
  if (dropdownMode === 'pathway' && codeSelect.value) {
    showLayer(pathKey(codeSelect.value));
  } else if (showingVerdict) {
    showVerdict();
  }
});

function onNodeClick(nodeId) {
  const mapped = CONFIG.nodeToLayer[nodeId];
  if (!mapped) {
    codeSelect.disabled = true; codeSelect.innerHTML = ''; dropdownMode = null;
    setStatus(nodeId + ': no dedicated layer');
    return;
  }
  if (Array.isArray(mapped)) {
    populateSubgroup(mapped, mapped[0]);
    showLayer(mapped[0]);
  } else {
    codeSelect.disabled = true; codeSelect.innerHTML = ''; dropdownMode = null;
    showLayer(mapped);
  }
}
function onTerminalClick(termId) {
  const info = CONFIG.terminalToCode[termId];
  if (!info) {
    codeSelect.disabled = true; codeSelect.innerHTML = ''; dropdownMode = null;
    setStatus(termId + ': no dedicated layer');
    return;
  }
  const codes = CONFIG.classCodes[info.verdict];
  populateCodes(codes, info.code);
  showLayer(pathKey(info.code));
}

let _activeNode = null;
const NODE_G = {};
let _activeSet = [];
function clearActiveSet() {
  _activeSet.forEach(g => { try { g.classList.remove('node-active'); } catch (e) { } });
  _activeSet = [];
}
function highlightTerminals(termIds) {
  try {
    if (_activeNode) { _activeNode.classList.remove('node-active'); _activeNode = null; }
    clearActiveSet();
    (termIds || []).forEach(tid => {
      const g = NODE_G[tid];
      if (g) { g.classList.add('node-active'); _activeSet.push(g); }
    });
    if (_activeSet.length) scrollTerminalIntoTreeView(_activeSet[0]);
  } catch (err) { }
}
function scrollTerminalIntoTreeView(g) {
  try {
    const pane = document.getElementById('treepane');
    const pr = pane.getBoundingClientRect();
    const gr = g.getBoundingClientRect();
    const fullyVisible = gr.top >= pr.top && gr.bottom <= pr.bottom &&
                         gr.left >= pr.left && gr.right <= pr.right;
    if (fullyVisible) return;
    if (typeof g.scrollIntoView === 'function') {
      g.scrollIntoView({ behavior: 'smooth', block: 'center', inline: 'center' });
    } else {
      pane.scrollTop += (gr.top - pr.top) - (pr.height - gr.height) / 2;
      pane.scrollLeft += (gr.left - pr.left) - (pr.width - gr.width) / 2;
    }
  } catch (e) { }
}
function setActiveNode(fromEl, boundG) {
  try {
    clearActiveSet();
    let el = fromEl;
    while (el && !(el.classList && el.classList.contains('node')) && el !== document) {
      el = el.parentNode;
    }
    const target = (el && el.classList && el.classList.contains('node')) ? el : boundG;
    if (!target) return;
    if (_activeNode && _activeNode !== target) _activeNode.classList.remove('node-active');
    target.classList.add('node-active');
    _activeNode = target;
  } catch (err) {
    try {
      if (boundG) {
        if (_activeNode && _activeNode !== boundG) _activeNode.classList.remove('node-active');
        boundG.classList.add('node-active');
        _activeNode = boundG;
      }
    } catch (e2) { }
  }
}

async function renderTree() {
  const { svg } = await mermaid.render('timberSvg', MERMAID_SRC);
  document.getElementById('treepane').innerHTML = svg;
  const groups = document.querySelectorAll('#treepane svg g.node, #treepane svg g[id]');
  groups.forEach(g => {
    const gid = g.id || '';
    for (const nodeId of CONFIG.allNodes) {
      if (gid.includes('-' + nodeId + '-') || gid === ('flowchart-' + nodeId)) {
        g.classList.add('node-clickable');
        NODE_G[nodeId] = g;
        g.addEventListener('click', (ev) => { onNodeClick(nodeId); setActiveNode(ev.target, g); });
      }
    }
    for (const termId of CONFIG.allTerminals) {
      if (gid.includes('-' + termId + '-') || gid === ('flowchart-' + termId)) {
        g.classList.add('node-clickable');
        NODE_G[termId] = g;
        g.addEventListener('click', (ev) => { onTerminalClick(termId); setActiveNode(ev.target, g); });
      }
    }
  });
  setTimeout(() => map.invalidateSize(), 200);
}
renderTree();
renderStats();
showLayer(null);  // verdict map OFF by default (click 'final verdict' to show)

document.getElementById('verdictBtn').addEventListener('click', () => {
  if (typeof _activeNode !== 'undefined' && _activeNode) { _activeNode.classList.remove('node-active'); _activeNode = null; }
  if (typeof clearActiveSet === 'function') clearActiveSet();
  showVerdict();
});

const bm = document.getElementById('bookmarks');
CONFIG.bookmarks.forEach(([name, lat, lon, z]) => {
  const b = document.createElement('button');
  b.textContent = name;
  b.title = name + ' (' + lat + ', ' + lon + ')';
  b.addEventListener('click', () => map.setView([lat, lon], z));
  bm.appendChild(b);
});

// --- Part 1/2: independent per-DATASET viz overlays (many on at once; never change the outcome) ---
map.createPane('dspane'); map.getPane('dspane').style.zIndex = 440;
function _esc(s){return String(s == null ? '' : s).replace(/[&<>"]/g, function(c){return {'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c];});}
const DATASET_TILE = {};
const DS_ON = new Set();
function _dsLayer(name) {
  if (!DATASET_TILE[name]) {
    const meta = CONFIG.datasetLayers[name];
    if (!meta) return null;
    DATASET_TILE[name] = L.tileLayer(meta.url, { pane: 'dspane', opacity: 0.9, maxZoom: 19 });
  }
  return DATASET_TILE[name];
}
function _dsCount() {
  const el = document.getElementById('dsCount');
  if (el) el.textContent = DS_ON.size;
}
function _dsSet(name, on) {  // add/remove one overlay and track its on-state (no parent sync here)
  const lyr = _dsLayer(name);
  if (!lyr) return;
  if (on) { lyr.addTo(map); DS_ON.add(name); } else { map.removeLayer(lyr); DS_ON.delete(name); }
}
function _dsRow(name) {
  const meta = CONFIG.datasetLayers[name] || { colour: '888888' };
  return '<label title="' + _esc(name) + '" style="display:block;line-height:1.6;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;cursor:pointer;padding-left:8px;">'
    + '<input type="checkbox" data-ds="' + name + '" style="vertical-align:middle;margin:0 3px 0 0;">'
    + '<span style="display:inline-block;width:9px;height:9px;border:1px solid #777;background:#'
    + meta.colour + ';margin:0 5px;vertical-align:middle;"></span>' + _esc(name) + '</label>';
}
function _parentCb() {  // tri-state parent checkbox; stopPropagation so it does not open/close the <details>
  return '<input type="checkbox" class="dsparent" title="toggle every layer in this group"'
    + ' onclick="event.stopPropagation()" style="margin-right:5px;vertical-align:middle;">';
}
function _dsSummary(text, count) {
  return '<summary style="cursor:pointer;line-height:1.8;">' + _parentCb()
    + '<span style="font-weight:600;">' + _esc(text) + '</span>'
    + (count != null ? ' <span style="color:#4b5563;font-weight:400;">(' + count + ')</span>' : '')
    + '</summary>';
}
function _groupHtml(grp) {
  var body = (grp.global || []).map(_dsRow).join('');
  if ((grp.national || []).length) {
    body += '<details style="margin:.1rem 0 .1rem 8px;border-left:2px solid #e5e7eb;padding-left:4px;">'
      + _dsSummary(grp.national_label || 'national', grp.national.length)
      + grp.national.map(_dsRow).join('') + '</details>';
  }
  var total = (grp.global || []).length + (grp.national || []).length;
  return '<details style="margin:.15rem 0;">' + _dsSummary(grp.label, total) + body + '</details>';
}
function _dsSyncParents() {  // reflect child state on every parent: checked / indeterminate / unchecked
  document.querySelectorAll('#datasetBody input.dsparent').forEach(function (p) {
    var kids = p.closest('details').querySelectorAll('input[data-ds]');
    var on = 0; kids.forEach(function (k) { if (k.checked) on++; });
    p.checked = on > 0 && on === kids.length;
    p.indeterminate = on > 0 && on < kids.length;
  });
}
function buildDatasetPanel() {
  var g = CONFIG.datasetGroups || { used: [], context: [] };
  var _hdr = 'font-weight:700;font-size:.7rem;text-transform:uppercase;letter-spacing:.03em;';
  var html = '<div style="margin:.2rem 0 .45rem;">'
    + '<button id="dsAllOff" type="button" style="font-size:.72rem;">all off</button>'
    + ' <span style="color:#4b5563;"><span id="dsCount" style="font-weight:700;color:#111827;">0</span> shown</span></div>';
  html += '<div style="' + _hdr + 'color:#374151;margin:.2rem 0;">Used in risk</div>';
  (g.used || []).forEach(function (grp) { html += _groupHtml(grp); });
  html += '<div style="' + _hdr + 'color:#4b5563;border-top:1px solid #e5e7eb;padding-top:.4rem;margin:.55rem 0 .2rem;">Not used for risk (context)</div>';
  (g.context || []).forEach(function (grp) { html += _groupHtml(grp); });
  var body = document.getElementById('datasetBody');
  if (!body) return;
  body.innerHTML = html;
  // individual member checkbox
  body.querySelectorAll('input[data-ds]').forEach(function (cb) {
    cb.addEventListener('change', function (e) {
      _dsSet(e.target.getAttribute('data-ds'), e.target.checked); _dsCount(); _dsSyncParents();
    });
  });
  // parent (group / national sub-group) checkbox: toggle every descendant member at once
  body.querySelectorAll('input.dsparent').forEach(function (p) {
    p.addEventListener('change', function (e) {
      var on = e.target.checked;
      e.target.closest('details').querySelectorAll('input[data-ds]').forEach(function (cb) {
        cb.checked = on; _dsSet(cb.getAttribute('data-ds'), on);
      });
      _dsCount(); _dsSyncParents();
    });
  });
  var off = document.getElementById('dsAllOff');
  if (off) off.addEventListener('click', function () {
    body.querySelectorAll('input[data-ds]').forEach(function (cb) {
      if (cb.checked) { cb.checked = false; _dsSet(cb.getAttribute('data-ds'), false); }
    });
    _dsCount(); _dsSyncParents();
  });
  _dsSyncParents();
}
function buildRefPanel() {
  var rows = CONFIG.referenceRows || [];
  var html = '';
  rows.forEach(function (r) {
    html += '<div style="margin-bottom:.4rem;">'
      + '<div style="font-weight:600;">' + _esc(r.question) + '</div>'
      + '<div style="color:#374151;">' + _esc(r.expr) + '</div>'
      + '<div style="color:#4b5563;">' + _esc((r.datasets || []).join(', ')) + '</div></div>';
  });
  var body = document.getElementById('refBody');
  if (body) body.innerHTML = html;
}
buildDatasetPanel();
buildRefPanel();

// =====================================================================================================
// DRAW A POLYGON -> RUN WHISP -> ATTACH ind_/risk_ COLUMNS + COLOUR THE POLYGON BY RISK
// -----------------------------------------------------------------------------------------------------
// Out of the box this targets the LIVE public Whisp API (set WHISP_API_KEY and it works). To use a dev
// instance instead, change WHISP_API_BASE (one line). The DRAW + PARSE + COLOUR logic works regardless;
// only the submit/poll network path needs a reachable API + key.
// CONFIG (edit these):
const WHISP_API_BASE = "https://whisp.openforis.org/api";  // live public API (routes under /api). For the
                                                           // dev instance, change THIS line to the dev base.
const WHISP_API_KEY  = "";                                 // <-- put your X-API-KEY here (blank = draw only, no analysis)
const RISK_COLUMN    = "risk_timber";                      // verdict column (low/high/more) , the fallback colour
const PATHWAY_COLUMN = "risk_timber_pathway";              // the SPECIFIC pathway label -> the exact map code colour

// risk_timber (low/high/more-info) -> the viewer's verdict map palette (the FALLBACK if the specific
// pathway label is unknown). Same hexes the map uses: low = green 41ab5d, more = amber f08c00, high = red e31a1c.
const RISK_COLOURS = { low: "#41ab5d", more: "#f08c00", high: "#e31a1c", unknown: "#6b7280" };
function riskToColour(val) {
  const v = String(val == null ? "" : val).toLowerCase();
  if (v.indexOf("high") >= 0) return RISK_COLOURS.high;
  if (v.indexOf("low") >= 0) return RISK_COLOURS.low;
  if (v.indexOf("more") >= 0 || v.indexOf("info") >= 0) return RISK_COLOURS.more;
  return RISK_COLOURS.unknown;
}

// risk_timber_pathway label (from add_risk_timber_col in src/openforis_whisp/risk.py) -> the viewer's
// pathway CODE. The code's colour comes from CONFIG.palette (the SAME _PALETTE_BY_CODE the map uses), so a
// drawn polygon gets the EXACT colour the map paints for that pathway. Codes/colours: see PALETTE in the
// build script. Code 14 is shared by the three "<class> 2020 -> other land" labels (as in the map).
const PATHWAY_LABEL_TO_CODE = CONFIG.pathwayLabelToCode;
function _normLabel(s) {  // tolerant match: lowercase, collapse whitespace, normalise the arrow + dashes
  return String(s == null ? "" : s).toLowerCase().trim()
    .replace(/\s+/g, " ").replace(/-+>/g, "->").replace(/→/g, "->");
}
const _PATHWAY_LOOKUP = {};
for (const k of Object.keys(PATHWAY_LABEL_TO_CODE)) _PATHWAY_LOOKUP[_normLabel(k)] = PATHWAY_LABEL_TO_CODE[k];
// Resolve a result's properties -> { code, colour, verdict } using the specific pathway label first,
// falling back to the plain risk_timber verdict colour if the label is missing / unrecognised.
function resolvePathwayStyle(props) {
  const label = props ? props[PATHWAY_COLUMN] : null;
  const code = label != null ? _PATHWAY_LOOKUP[_normLabel(label)] : undefined;
  const verdict = props ? props[RISK_COLUMN] : null;
  if (code !== undefined && CONFIG.palette && CONFIG.palette[code]) {
    return { code: code, colour: "#" + CONFIG.palette[code], verdict: verdict, label: label };
  }
  return { code: null, colour: riskToColour(verdict), verdict: verdict, label: label };  // fallback
}

// Drawn polygons live in their own feature group, above the overlays.
map.createPane('drawpane'); map.getPane('drawpane').style.zIndex = 650;
const drawnItems = new L.FeatureGroup();
map.addLayer(drawnItems);
const drawControl = new L.Control.Draw({
  edit: { featureGroup: drawnItems, edit: false, remove: true },
  draw: { polygon: { allowIntersection: false, showArea: true }, polyline: false, rectangle: false,
          circle: false, circlemarker: false, marker: false },
});
map.addControl(drawControl);

function _drawStatus(msg) { setStatus(msg); }

// Collect every ind_/Ind_ or risk_ property (case-insensitive) from a result feature's properties.
function collectIndRisk(props) {
  const out = {};
  if (!props) return out;
  for (const k of Object.keys(props)) {
    const lk = k.toLowerCase();
    if (lk.startsWith("ind_") || lk.startsWith("risk_")) out[k] = props[k];
  }
  return out;
}

// Style a drawn polygon by its SPECIFIC pathway code colour (fallback: the plain risk_timber colour).
function styleByRisk(layer, props) {
  const colour = props ? resolvePathwayStyle(props).colour : RISK_COLOURS.unknown;
  layer.setStyle({ color: colour, weight: 2, fillColor: colour, fillOpacity: 0.45 });
}

// Build a popup: a header line "code N: <pathway label> (LOW/HIGH/MORE-INFO)" then the ind_/risk_ values.
// ---- decision-story popup: pretty timber pathway, faithful to risk.py add_risk_timber_col ----
// Takes the Whisp result properties for ONE polygon and renders the decision PATH it took
// (each 2020-state question, the answer, and the leg that fired) plus the outcome. Degrades
// gracefully: a missing column reads as "no", and the headline badge uses the server's
// risk_timber when present (authoritative) so it stays correct even on the old public-API schema.
const STORY_COLORS = { low: '#2e7d32', more_info_needed: '#ef6c00', high: '#c62828' };
__TREEJS__

function decisionStoryHtml(p, style) {
  p = p || {};
  const story = timberStory(p);
  const serverResult = (p['risk_timber'] || story.result || '').toString();
  const col = STORY_COLORS[serverResult] || (style && style.colour) || '#666';
  const pathway = p['risk_timber_pathway'] || '';
  const resultTxt = serverResult.replace(/_/g, ' ').toUpperCase() || 'n/a';
  let html = '<div style="font-family:sans-serif;min-width:235px;max-width:310px">';
  html += '<div style="font-weight:600;margin-bottom:4px">Timber pathway</div>';
  html += '<div style="padding:4px 8px;border-radius:4px;color:#fff;background:' + col + ';font-weight:600;margin-bottom:6px">'
    + resultTxt + (pathway ? ' &middot; ' + pathway : '') + '</div>';
  html += '<div style="font-size:11px;line-height:1.55">';
  story.steps.forEach((s) => {
    const mark = s.ans === null ? '&bull;' : (s.ans ? '&#10003;' : '&#10007;');
    const markCol = s.ans === null ? '#888' : (s.ans ? (s.fired ? col : '#1565c0') : '#bbb');
    const wrap = s.fired
      ? 'border-left:3px solid ' + col + ';padding-left:6px;margin:3px 0;font-weight:600'
      : 'padding-left:9px;margin:3px 0;color:#555';
    html += '<div style="' + wrap + '"><span style="color:' + markCol + ';font-weight:700">' + mark + '</span> ' + s.q;
    if (s.fired && s.leg) html += '<div style="margin-left:15px;color:' + col + '">&rarr; ' + s.leg + '</div>';
    html += '</div>';
  });
  html += '</div>';
  const indKeys = Object.keys(p).filter((k) => /^ind_|^risk_|^primary_2025$/i.test(k)).sort();
  if (indKeys.length) {
    html += '<details style="margin-top:6px;font-size:10px"><summary style="cursor:pointer;color:#1565c0">all indicator values</summary>'
      + '<div style="max-height:140px;overflow:auto;margin-top:3px">';
    indKeys.forEach((k) => { html += '<div>' + k + ': ' + p[k] + '</div>'; });
    html += '</div></details>';
  }
  html += '</div>';
  return html;
}
function indRiskPopupHtml(indRisk, style) {
  const verdict = style && style.verdict != null ? String(style.verdict).replace(/_/g, " ").toUpperCase() : "";
  let header = "<b>Whisp result</b>";
  if (style) {
    const codeTxt = style.code != null ? ("code " + style.code + ": ") : "";
    const labelTxt = style.label != null ? style.label : (style.verdict != null ? style.verdict : "n/a");
    header += "<div style='margin-top:2px'><span style='display:inline-block;width:11px;height:11px;"
      + "border:1px solid #777;background:" + style.colour + ";margin-right:5px;vertical-align:middle'></span>"
      + codeTxt + labelTxt + (verdict ? " (" + verdict + ")" : "") + "</div>";
  }
  const keys = Object.keys(indRisk);
  if (!keys.length) return header + "<br/><span style='font-size:11px'>(no ind_/risk_ columns found)</span>";
  const riskFirst = keys.sort((p, q) => {
    const rp = p.toLowerCase().startsWith("risk_") ? 0 : 1;
    const rq = q.toLowerCase().startsWith("risk_") ? 0 : 1;
    return rp - rq || p.localeCompare(q);
  });
  let html = header + "<div style='max-height:200px;overflow:auto;font-size:11px;margin-top:4px'>";
  for (const k of riskFirst) html += "<div><b>" + k + "</b>: " + indRisk[k] + "</div>";
  html += "</div>";
  return html;
}

// Apply a result feature (its properties) to the drawn layer: attach ind_/risk_, colour by pathway, popup.
function applyResultToLayer(layer, resultProps) {
  const indRisk = collectIndRisk(resultProps);
  layer.feature = layer.feature || { type: "Feature", properties: {} };
  layer.feature.properties = Object.assign({}, layer.feature.properties, indRisk);
  const style = resolvePathwayStyle(layer.feature.properties);
  layer.setStyle({ color: style.colour, weight: 2, fillColor: style.colour, fillOpacity: 0.45 });
  layer.bindPopup(decisionStoryHtml(layer.feature.properties, style)).openPopup();
  _drawStatus("Whisp result: " + (style.code != null ? "code " + style.code + " " : "")
    + (style.label != null ? style.label : (style.verdict != null ? style.verdict : "n/a"))
    + "  (" + Object.keys(indRisk).length + " ind_/risk_ columns attached)");
}

// Extract the FIRST feature from a result GeoJSON (FeatureCollection or Feature or array of rows).
function firstResultProps(result) {
  if (!result) return null;
  if (Array.isArray(result) && result.length) return result[0].properties || result[0];
  if (result.type === "FeatureCollection" && result.features && result.features.length)
    return result.features[0].properties || {};
  if (result.type === "Feature") return result.properties || {};
  if (result.features && result.features.length) return result.features[0].properties || {};
  if (result.data) return firstResultProps(result.data);  // unwrap an envelope-in-envelope
  return result.properties || result;  // last resort: treat the object itself as the props bag
}

// Poll GET /status/{token} every ~3s until analysis_completed (or an error / ~2 min timeout).
async function pollStatus(token) {
  const deadline = Date.now() + 120000;  // ~2 min
  while (Date.now() < deadline) {
    await new Promise(r => setTimeout(r, 3000));
    const resp = await fetch(WHISP_API_BASE + "/status/" + token, { headers: { "X-API-KEY": WHISP_API_KEY } });
    const env = await resp.json().catch(() => ({}));
    const code = env && env.code;
    _drawStatus("Whisp analysis: " + (code || "running") + " ...");
    if (code === "analysis_completed") return env;
    if (code && /error|fail/i.test(code)) throw new Error("Whisp analysis returned: " + code + (env.message ? " (" + env.message + ")" : ""));
  }
  throw new Error("Whisp analysis timed out after ~2 minutes.");
}

// Fetch the result GeoJSON for a token: prefer inline data; else GET /generate-geojson/{token} (no key).
async function fetchResultGeojson(env, token) {
  if (env && env.data && (env.data.type || env.data.features || Array.isArray(env.data))) return env.data;
  const resp = await fetch(WHISP_API_BASE + "/generate-geojson/" + token);
  return await resp.json();
}

// Submit a drawn polygon's FeatureCollection to the Whisp API, poll, attach + colour the result.
async function analyzeDrawnLayer(layer, featureCollection) {
  if (!WHISP_API_KEY) {
    layer.bindPopup("Drawn polygon. Set WHISP_API_KEY (top of the script) to analyze it with Whisp.").openPopup();
    _drawStatus("Drawn polygon added. Set WHISP_API_KEY to analyze.");
    return;
  }
  try {
    _drawStatus("Submitting polygon to Whisp ...");
    // Body = the FeatureCollection with an analysisOptions object merged at top level (mirrors
    // whisp-app's useSubmitAnalysis.ts). nationalCodes/generateGeoids are sensible defaults.
    const body = Object.assign({}, featureCollection, {
      analysisOptions: { nationalCodes: [], generateGeoids: false, unitType: "ha" },
    });
    const resp = await fetch(WHISP_API_BASE + "/submit/geojson", {
      method: "POST",
      headers: { "X-API-KEY": WHISP_API_KEY, "Content-Type": "application/json" },
      body: JSON.stringify(body),
    });
    const env = await resp.json().catch(() => ({}));   // envelope: { code, message, data }
    let completedEnv = env;
    let token = (env && env.data && env.data.token) ? env.data.token : (env && env.token);
    if (!env || env.code !== "analysis_completed") {
      if (!token) throw new Error("Whisp submit did not return a token or inline result"
        + (env && env.message ? " (" + env.message + ")" : ""));
      completedEnv = await pollStatus(token);   // poll until analysis_completed / error / timeout
    }
    const result = await fetchResultGeojson(completedEnv, token);
    const props = firstResultProps(result);
    if (!props) throw new Error("Whisp result had no feature properties");
    applyResultToLayer(layer, props);
  } catch (err) {
    _drawStatus("Whisp analysis failed: " + (err && err.message ? err.message : err));
    layer.bindPopup("Whisp analysis failed:<br/>" + (err && err.message ? err.message : err)).openPopup();
  }
}

// draw:created -> add the polygon, build its FeatureCollection, run (or prompt for a key).
map.on(L.Draw.Event.CREATED, function (e) {
  const layer = e.layer;
  layer.options.pane = 'drawpane';  // keep drawn polygons above the raster overlays
  drawnItems.addLayer(layer);
  styleByRisk(layer, null);  // neutral until a result comes back
  const feature = layer.toGeoJSON();   // a GeoJSON Feature (Polygon)
  feature.properties = feature.properties || {};
  const featureCollection = { type: "FeatureCollection", features: [feature] };
  analyzeDrawnLayer(layer, featureCollection);
});
map.on(L.Draw.Event.DELETED, function () { _drawStatus("Drawn polygon(s) removed."); });

// ----- FEATURE C: draggable split slider between the tree pane (left) and the map pane (right) -----
(function () {
  const splitter = document.getElementById('splitter');
  const treepane = document.getElementById('treepane');
  const wrap = document.querySelector('.wrap');
  const MIN_TREE = 200;
  const MIN_MAP = 260;
  let dragging = false;
  function onMove(e) {
    if (!dragging) return;
    const rect = wrap.getBoundingClientRect();
    let w = e.clientX - rect.left;
    const maxTree = rect.width - splitter.offsetWidth - MIN_MAP - (document.getElementById('controlspane') ? document.getElementById('controlspane').offsetWidth : 0);
    if (w < MIN_TREE) w = MIN_TREE;
    if (w > maxTree) w = maxTree;
    treepane.style.width = w + 'px';
    map.invalidateSize();
  }
  function onUp() {
    if (!dragging) return;
    dragging = false;
    document.body.classList.remove('dragging');
    document.removeEventListener('mousemove', onMove);
    document.removeEventListener('mouseup', onUp);
    map.invalidateSize();
  }
  splitter.addEventListener('mousedown', (e) => {
    e.preventDefault();
    dragging = true;
    document.body.classList.add('dragging');
    document.addEventListener('mousemove', onMove);
    document.addEventListener('mouseup', onUp);
  });
})();
</script>
</body>
</html>
"""

html = (HTML_TEMPLATE
        .replace("__CONFIG__", json.dumps(_JS_CONFIG))
        .replace("__MERMAID__", json.dumps(MERMAID_SRC))
        .replace("__TREEJS__", tx.js_module_source()))
print("Assembled HTML: %d chars, %d overlay layers, %d Sentinel-2 backgrounds." % (
    len(html), len(registry), len(S2_BACKGROUNDS)))

Assembled HTML: 116622 chars, 57 overlay layers, 2 Sentinel-2 backgrounds.


## Render the viewer

The viewer is shown inline below inside a fixed-height iframe (so the embedded Leaflet / Mermaid scripts run in their own document), and is also written to `timber_pathway_viewer.html`, which is downloaded to your machine (in Colab) or reported as a local path otherwise.

> **If the panels are blank inline** (some environments, e.g. VSCode, strip embedded scripts): open the saved `timber_pathway_viewer.html` in a browser - it renders the full viewer.

In [9]:
import html as _html

# Inject the Whisp API key from the WHISP_API_KEY environment variable at RUNTIME only.
# The committed notebook keeps the key BLANK. Set it in your session BEFORE running this cell:
#     import os; os.environ['WHISP_API_KEY'] = 'your-key'
import os
_wk = os.environ.get('WHISP_API_KEY', '')
if not _wk:
    # fallback: a gitignored key file (one line = your key). Looks in CWD, repo root, notebooks/, and home.
    import pathlib
    for _cand in ('.whisp_api_key', '../.whisp_api_key', 'notebooks/.whisp_api_key', str(pathlib.Path.home() / '.whisp_api_key')):
        _kp = pathlib.Path(_cand)
        if _kp.exists():
            _wk = _kp.read_text(encoding='utf-8').strip()
            print('WHISP_API_KEY read from', _cand)
            break
if _wk:
    html = html.replace('const WHISP_API_KEY  = ""', 'const WHISP_API_KEY  = "%s"' % _wk)
    print('WHISP_API_KEY injected from environment (kept out of the saved notebook).')
else:
    print('No WHISP_API_KEY (env var or .whisp_api_key file); draw works, analysis disabled.')

OUTPUT_HTML = 'timber_pathway_viewer.html'
with open(OUTPUT_HTML, 'w', encoding='utf-8') as f:
    f.write(html)
print('Wrote', OUTPUT_HTML, '(' + str(len(html)) + ' chars)')
print('Tile layers embedded:', len(registry), 'overlays +', len(S2_BACKGROUNDS), 'S2 backgrounds')

# Show inline inside a FIXED-HEIGHT iframe via srcdoc, so the embedded Leaflet / Mermaid scripts
# run in their own document and the panes get a real height (a bare display(HTML(...)) gets its
# scripts stripped and its containers collapsed in some notebook sandboxes, e.g. VSCode).
_iframe = (
    '<iframe srcdoc="' + _html.escape(html)
    + '" style="width:100%;height:850px;border:0;"></iframe>'
)
display(HTML(_iframe))

# Deliver the standalone file to the user's machine, matching the other Whisp example notebooks
# (Colab_whisp_geojson_to_csv.ipynb uses `from google.colab import files; files.download(path)`).
# Guarded so it no-ops gracefully outside Colab (just reports the local saved path).
try:
    from google.colab import files  # type: ignore
    files.download(OUTPUT_HTML)
    print('Download started:', OUTPUT_HTML)
except ImportError:
    import os
    print('Not running in Colab; open the saved file from:', os.path.abspath(OUTPUT_HTML))

WHISP_API_KEY read from ../.whisp_api_key
WHISP_API_KEY injected from environment (kept out of the saved notebook).
Wrote timber_pathway_viewer.html (116658 chars)
Tile layers embedded: 57 overlays + 2 S2 backgrounds


Not running in Colab; open the saved file from: c:\Users\Arnell\Documents\GitHub\whisp\notebooks\timber_pathway_viewer.html


## Run in a real browser with working analysis (local serve + proxy)

The draw feature can't call the Whisp API from this notebook's inline preview: the public API blocks
cross-origin browser calls (CORS) and the VSCode preview adds its own CSP. The cell below works around
both by serving the built viewer from a tiny local server that also **proxies** `/api/*` to the Whisp
API, adding your `X-API-KEY` **server-side** (the key never reaches the browser). Page and API are then
same-origin, so there's no CORS, and it's a real browser tab, so there's no CSP block.

Run the build + render cells first, then run this cell and open the printed URL in **Chrome/Edge**
(not the VSCode preview). It proxies to the **public** API by default (released whisp); point
`PROXY_UPSTREAM` at your dev instance to exercise this branch.


In [10]:
# Local serve + proxy: run the viewer in a REAL browser with WORKING analysis.
# Same-origin (no CORS) + real browser (no VSCode CSP). The proxy only accepts same-origin calls
# to the viewer's 3 routes. Pick the analysis backend with BACKEND below:
#   "package" = run the whisp PACKAGE locally on YOUR Earth Engine project (no hosted API, no key).
#               Immune to the public API being down; uses the installed openforis_whisp (PyPI or branch).
#   "api"     = relay /api/* to a hosted Whisp API (PROXY_UPSTREAM), adding your X-API-KEY server-side.
#   "auto"    = try "api" first, fall back to "package" on any error.
# Change BACKEND and RE-RUN this cell to switch (the running server reads it live; no kernel restart).
import os, pathlib, threading, json, tempfile
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import requests

PROXY_PORT = 8787
BACKEND = "package"          # "package" | "api" | "auto"
PKG_NATIONAL_CODES = ["br"]  # package mode: Brazil layers ON; use [] for global-only

if os.environ.get('WHISP_VIEWER_BUILD') == '1':
    print('(build mode: proxy server not started)')
else:
    _proxy_html = pathlib.Path(OUTPUT_HTML if 'OUTPUT_HTML' in globals() else 'timber_pathway_viewer.html')
    _html_text = _proxy_html.read_text(encoding='utf-8')
    _q = chr(34)
    if 'const WHISP_API_BASE' in _html_text:
        _api_base = _html_text.split('const WHISP_API_BASE', 1)[1].split(_q, 2)[1]
    else:
        _api_base = 'https://whisp.openforis.org/api'
    PROXY_UPSTREAM = _api_base.rsplit('/api', 1)[0]   # host only; follows the viewer's WHISP_API_BASE

    _proxy_key = os.environ.get('WHISP_API_KEY', '')
    if not _proxy_key:
        for _c in ('.whisp_api_key', '../.whisp_api_key', 'notebooks/.whisp_api_key', str(pathlib.Path.home() / '.whisp_api_key')):
            if pathlib.Path(_c).exists():
                _proxy_key = pathlib.Path(_c).read_text(encoding='utf-8').strip(); break

    _allow_origins = ('http://localhost:%d' % PROXY_PORT, 'http://127.0.0.1:%d' % PROXY_PORT)
    _allow_hosts = ('localhost:%d' % PROXY_PORT, '127.0.0.1:%d' % PROXY_PORT)
    _allow_routes = ('/api/submit/geojson', '/api/status/', '/api/generate-geojson/')

    # ---- package backend: run the whisp PACKAGE on the posted polygon, return the viewer's envelope ----
    # The package (risk.py) is the single source of truth; no tree/pathway logic is reimplemented here.
    # whisp_risk's dataframe is converted to GeoJSON via the package's own geometry-column serialiser
    # (convert_df_to_geojson, geo_column="geo"), so features[0].properties carries risk_timber +
    # risk_timber_pathway + the Ind_* columns - exactly what the viewer JS reads.
    _pkg_img = None
    def _pkg_image():
        global _pkg_img
        if _pkg_img is None:
            import openforis_whisp as _w
            print('  [package] building the whisp image once (a few seconds)...')
            _pkg_img = _w.combine_datasets(national_codes=PKG_NATIONAL_CODES, auto_recovery=True)
        return _pkg_img

    def _run_package(body_bytes):
        import openforis_whisp as _w
        fc = json.loads(body_bytes.decode('utf-8'))
        feats = fc.get('features', [])
        for i, ft in enumerate(feats):
            ft.setdefault('properties', {})
            ft['properties']['external_id'] = ft['properties'].get('external_id') or ('drawn_%d' % (i + 1))
        clean = {'type': 'FeatureCollection', 'features': feats}
        with tempfile.NamedTemporaryFile('w', suffix='.geojson', delete=False) as _f:
            json.dump(clean, _f); _in = _f.name
        df = _w.whisp_formatted_stats_geojson_to_df(
            input_geojson_filepath=_in, external_id_column='external_id',
            national_codes=PKG_NATIONAL_CODES, unit_type='ha',
            whisp_image=_pkg_image(), mode='sequential')
        df = _w.whisp_risk(df, national_codes=PKG_NATIONAL_CODES)
        _out = tempfile.NamedTemporaryFile('w', suffix='.geojson', delete=False).name
        _w.convert_df_to_geojson(df, _out, geo_column='geo')
        result_fc = json.loads(pathlib.Path(_out).read_text(encoding='utf-8'))
        return {'code': 'analysis_completed', 'data': result_fc}

    def _proxy_page():
        # serve the built viewer, but (1) repoint its API base at this server (same-origin) and
        # (2) set a placeholder key so the page's `if (!WHISP_API_KEY) return` guard passes -
        # the real key is added by the proxy, never sent to the browser. Both edits are scoped
        # to the const declaration line so other text/URLs are untouched.
        t = _proxy_html.read_text(encoding='utf-8')
        for _name, _val in (('const WHISP_API_BASE', '/api'), ('const WHISP_API_KEY', 'proxy')):
            if _name in t:
                _pre, _rest = t.split(_name, 1)
                _decl, _after = _rest.split(';', 1)
                t = _pre + _name + _decl.split('=')[0] + '= ' + _q + _val + _q + ';' + _after
        return t.encode('utf-8')

    def _send_json(handler, obj, status=200):
        b = json.dumps(obj).encode('utf-8')
        handler.send_response(status)
        handler.send_header('Content-Type', 'application/json')
        handler.send_header('Content-Length', str(len(b))); handler.end_headers()
        handler.wfile.write(b)

    class _ProxyHandler(BaseHTTPRequestHandler):
        def _local_only(self):
            o = self.headers.get('Origin')
            if o and o not in _allow_origins:   # block cross-origin browser abuse of the key
                return False
            return self.headers.get('Host', _allow_hosts[0]) in _allow_hosts   # blunt DNS rebinding
        def do_OPTIONS(self):
            self.send_response(204); self.end_headers()
        def do_GET(self):
            if self.path.startswith('/api/'): return self._proxy('GET')
            if self.path in ('/', '/index.html'):
                if not self._local_only(): self.send_response(403); self.end_headers(); return
                b = _proxy_page()
                self.send_response(200); self.send_header('Content-Type', 'text/html; charset=utf-8')
                self.send_header('Content-Length', str(len(b))); self.end_headers(); self.wfile.write(b); return
            self.send_response(404); self.end_headers()
        def do_POST(self):
            if self.path.startswith('/api/'): return self._proxy('POST')
            self.send_response(404); self.end_headers()
        def _proxy(self, method):
            if not self._local_only():
                self.send_response(403); self.end_headers(); return
            if not any(self.path == r or self.path.startswith(r) for r in _allow_routes):
                self.send_response(404); self.end_headers(); return
            n = int(self.headers.get('Content-Length', 0) or 0)
            body = self.rfile.read(n) if n else None
            is_submit = method == 'POST' and self.path.startswith('/api/submit')
            if is_submit and BACKEND == 'package':
                try:
                    return _send_json(self, _run_package(body))
                except Exception as e:
                    return _send_json(self, {'code': 'error', 'message': 'package: %r' % e})
            if is_submit and BACKEND == 'auto':
                try:
                    return self._relay(method, body)
                except Exception:
                    try:
                        return _send_json(self, _run_package(body))
                    except Exception as e:
                        return _send_json(self, {'code': 'error', 'message': 'auto: %r' % e})
            # api mode, or the non-submit routes (status / generate-geojson): relay upstream
            try:
                return self._relay(method, body)
            except Exception as e:
                self.send_response(502); self.end_headers(); self.wfile.write(str(e).encode()); return
        def _relay(self, method, body):
            hdr = {}
            if _proxy_key: hdr['X-API-KEY'] = _proxy_key   # real key added here, server-side only
            if self.headers.get('Content-Type'): hdr['Content-Type'] = self.headers['Content-Type']
            r = requests.request(method, PROXY_UPSTREAM + self.path, headers=hdr, data=body, timeout=120)
            self.send_response(r.status_code)
            self.send_header('Content-Type', r.headers.get('Content-Type', 'application/json'))
            self.send_header('Content-Length', str(len(r.content))); self.end_headers()
            self.wfile.write(r.content)
        def log_message(self, *a): pass

    # ALWAYS (re)start, so re-running this cell picks up BACKEND / PROXY_PORT / code changes.
    # (The previous "don't restart if already running" guard meant changes were silently ignored
    #  and the OLD server kept serving - e.g. relaying to the hosted API instead of the package.)
    try:
        _proxy_srv.shutdown(); _proxy_srv.server_close()
    except Exception:
        pass
    try:
        _proxy_srv = ThreadingHTTPServer(('127.0.0.1', PROXY_PORT), _ProxyHandler)
        threading.Thread(target=_proxy_srv.serve_forever, daemon=True).start()
        print('Serving at http://127.0.0.1:%d/   backend=%s' % (PROXY_PORT, BACKEND))
    except OSError as _e:
        print('Port %d is busy (%s) - set PROXY_PORT to a free port and re-run this cell.' % (PROXY_PORT, _e))
    print('Open  http://127.0.0.1:%d/  in Chrome/Edge (use 127.0.0.1, NOT localhost; NOT the VSCode preview).' % PROXY_PORT)
    if BACKEND == 'package':
        print('Backend = PACKAGE: analysis runs locally via openforis_whisp on your EE (NO hosted API).')
    else:
        print('Backend = %s -> relay to %s  (key %s).' % (BACKEND, PROXY_UPSTREAM, 'loaded' if _proxy_key else 'MISSING'))


Serving at http://127.0.0.1:8787/   backend=package
Open  http://127.0.0.1:8787/  in Chrome/Edge (use 127.0.0.1, NOT localhost; NOT the VSCode preview).
Backend = PACKAGE: analysis runs locally via openforis_whisp on your EE (NO hosted API).
